# SpeakAI-Eval — Setup Environment on Kaggle
**Yêu cầu:** Thêm dataset `tnguynthnh142/speakai-models` vào Notebook.


In [ ]:
import os
TARGET_DIR = '/kaggle/working/SpeakAI-Eval'
os.makedirs(f'{TARGET_DIR}/configs', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/data', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/models', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/infer', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/speaker-diarize/speaker_diarize', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/pretrained_models', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/transformer_models', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/hf_cache', exist_ok=True)
print('Directories created on Kaggle.')

---
### Tải Source Code (Offline)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/paths.py
"""Project root paths — import from anywhere after sys.path includes repo root."""

from __future__ import annotations

from pathlib import Path

ROOT = Path(__file__).resolve().parent
CONFIGS = ROOT / "configs"
PRONUNCIATION_CONFIG = CONFIGS / "pronunciation.yaml"

TRANSFORMER_MODELS = ROOT / "transformer_models"
SPEAKER_DIARIZE_DIR = ROOT / "speaker-diarize"
LOGS_DIR = ROOT / "logs"
WEB_UPLOADS = ROOT / "web" / "uploads"
TEACHER_REFERENCE_DIR = WEB_UPLOADS / "teacher_reference"


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/configs', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/configs/pronunciation.yaml
# Pronunciation Assessment — Kaggle Edition (GPU only, no HuggingFace auto-download)
project:
  seed: 42
  device: cuda

paths:
  transformer_models_dir: transformer_models
  pronunciation_checkpoint: transformer_models/pronunciation.pt
  checkpoint_dir: checkpoints
  log_dir: logs
  cmudict_path: null

audio_preprocess:
  sample_rate: 16000
  peak_normalize: true
  target_peak: 0.95
  highpass_hz: 80.0
  denoise: true
  denoise_prop: 0.75
  speech_normalize: true
  speech_thresh_db: -35.0
  vad_frame_ms: 25.0
  vad_hop_ms: 10.0

inference:
  device: cuda
  max_audio_duration_sec: 3600
  max_duration_sec: null
  max_upload_mb: 300

asr:
  model_name: pretrained_models/whisper-large-v3-turbo
  language: en
  device: cuda
  torch_dtype: float16
  max_new_tokens: 440
  lang_id:
    enabled: true
    drop_languages: [vi, vie, vietnamese]
    min_confidence: 0.45
    text_vi_regex: true

wavlm:
  model_name: pretrained_models/wavlm-large
  freeze: true
  use_lora: false

transformer:
  num_layers: 3
  num_heads: 8
  ff_dim: 2048
  dropout: 0.1
  max_seq_len: 800

ctc_align:
  use_wavlm_ctc_head: true

phoneme_graph:
  hidden_dim: 256
  num_gat_layers: 2
  num_heads: 4
  dropout: 0.1
  edge_types:
    sequential: true
    same_word: true
    same_syllable: false

multitask:
  hidden_dim: 256
  dropout: 0.1
  utterance_aspects: [accuracy, fluency, completeness, prosodic, total]
  word_aspects: [accuracy, stress, total]
  phoneme_aspects: [accuracy]
  score_scale: 5.0

scorer:
  weights:
    utterance_total: 0.4
    word_total: 0.3
    phoneme_accuracy: 0.3
  phoneme_low_threshold: 1.2

sentence_split:
  min_silence_sec: 0.35
  silence_thresh_db: -40
  min_segment_sec: 0.2
  max_segment_sec: null
  padding_sec: 0.1
  trim_edges: false
  trim_min_sec: 0.05
  diarization_merge_gap_sec: 0.2


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/audio_preprocess.py
"""Shared audio preprocessing: resample → high-pass → denoise → speech-aware peak normalize."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple, Union

import torch
import torchaudio

PathLike = Union[str, Path]


def load_audio_file(path: PathLike) -> Tuple[torch.Tensor, int]:
    """Load audio as float tensor [channels, samples] without torchcodec."""
    import soundfile as sf

    data, sr = sf.read(str(path), dtype="float32")
    if data.ndim == 1:
        wav = torch.from_numpy(data).unsqueeze(0)
    else:
        wav = torch.from_numpy(data.T.copy())
    return wav, int(sr)


def save_audio_file(path: PathLike, wav: torch.Tensor, sample_rate: int) -> None:
    """Save audio tensor [channels, samples] without torchcodec."""
    import soundfile as sf

    x = wav.detach().cpu()
    if x.ndim == 1:
        data = x.numpy()
    elif x.ndim == 2:
        data = x.T.numpy()
    else:
        raise ValueError(f"Unexpected wav shape {tuple(x.shape)}")
    sf.write(str(path), data, sample_rate, subtype="FLOAT")


@dataclass
class PreprocessConfig:
    sample_rate: int = 16000
    peak_normalize: bool = True
    target_peak: float = 0.95
    highpass_hz: float = 80.0
    denoise: bool = False
    denoise_prop: float = 0.75
    speech_normalize: bool = True
    speech_thresh_db: float = -35.0
    vad_frame_ms: float = 25.0
    vad_hop_ms: float = 10.0

    @classmethod
    def from_dict(cls, cfg: Optional[dict]) -> "PreprocessConfig":
        if not cfg:
            return cls()
        fields = cls.__dataclass_fields__
        return cls(**{k: v for k, v in cfg.items() if k in fields})


def _to_mono(wav: torch.Tensor) -> torch.Tensor:
    return wav.mean(0) if wav.dim() > 1 else wav


def _resample(wav: torch.Tensor, sr: int, target_sr: int) -> torch.Tensor:
    if sr == target_sr:
        return wav
    return torchaudio.functional.resample(wav, sr, target_sr)


def _highpass(wav: torch.Tensor, sr: int, cutoff: float) -> torch.Tensor:
    if cutoff <= 0:
        return wav
    return torchaudio.functional.highpass_biquad(wav.unsqueeze(0), sr, cutoff).squeeze(0)


def frame_rms(
    wav: torch.Tensor,
    sample_rate: int,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
) -> torch.Tensor:
    frame_len = max(1, int(sample_rate * frame_ms / 1000))
    hop = max(1, int(sample_rate * hop_ms / 1000))
    if wav.numel() < frame_len:
        return torch.tensor([wav.pow(2).mean().sqrt()])
    frames = wav.unfold(0, frame_len, hop)
    return torch.sqrt(frames.pow(2).mean(dim=1) + 1e-10)


def speech_frame_mask(
    wav: torch.Tensor,
    sample_rate: int,
    *,
    thresh_db: float = -35.0,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
) -> torch.Tensor:
    """Boolean mask per analysis frame (True = speech)."""
    if wav.numel() == 0:
        return torch.zeros(0, dtype=torch.bool)
    rms = frame_rms(wav, sample_rate, frame_ms, hop_ms)
    ref = rms.max().clamp(min=1e-8)
    thresh = ref * (10 ** (thresh_db / 20.0))
    return rms >= thresh


def speech_sample_mask(
    wav: torch.Tensor,
    sample_rate: int,
    *,
    thresh_db: float = -35.0,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
) -> torch.Tensor:
    """Expand frame-level speech mask to per-sample boolean mask."""
    n = wav.shape[0]
    if n == 0:
        return torch.zeros(0, dtype=torch.bool)
    frame_len = max(1, int(sample_rate * frame_ms / 1000))
    hop = max(1, int(sample_rate * hop_ms / 1000))
    frames = speech_frame_mask(wav, sample_rate, thresh_db=thresh_db, frame_ms=frame_ms, hop_ms=hop_ms)
    mask = torch.zeros(n, dtype=torch.bool)
    for i, is_speech in enumerate(frames):
        start = i * hop
        end = min(n, start + frame_len)
        if is_speech:
            mask[start:end] = True
    return mask


def trim_silence_edges(
    wav: torch.Tensor,
    sample_rate: int,
    *,
    thresh_db: float = -35.0,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
    min_samples: int = 0,
) -> Tuple[torch.Tensor, int, int]:
    """Trim leading/trailing non-speech. Returns (clip, start_offset, end_offset) in samples."""
    n = wav.shape[0]
    if n == 0:
        return wav, 0, 0

    frames = speech_frame_mask(wav, sample_rate, thresh_db=thresh_db, frame_ms=frame_ms, hop_ms=hop_ms)
    speech_idx = torch.nonzero(frames, as_tuple=False).flatten()
    if speech_idx.numel() == 0:
        return wav, 0, 0

    hop = max(1, int(sample_rate * hop_ms / 1000))
    frame_len = max(1, int(sample_rate * frame_ms / 1000))
    first = int(speech_idx[0].item()) * hop
    last = min(n, int(speech_idx[-1].item()) * hop + frame_len)

    if last - first < min_samples:
        return wav, 0, 0
    return wav[first:last].contiguous(), first, n - last


def _peak_normalize(
    wav: torch.Tensor,
    target: float,
    speech_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    if speech_mask is not None and speech_mask.any():
        peak = wav[speech_mask].abs().max()
    else:
        peak = wav.abs().max()
    if float(peak) < 1e-8:
        return wav
    return wav * (target / peak)


def _denoise(wav: torch.Tensor, sr: int, prop: float) -> torch.Tensor:
    try:
        import noisereduce as nr

        y = nr.reduce_noise(y=wav.detach().cpu().numpy(), sr=sr, stationary=True, prop_decrease=prop)
        return torch.from_numpy(y).to(dtype=wav.dtype, device=wav.device)
    except ImportError:
        return wav


def preprocess_waveform(
    wav: torch.Tensor,
    sample_rate: int,
    cfg: Optional[PreprocessConfig] = None,
) -> torch.Tensor:
    """Apply full chain to a mono waveform tensor."""
    cfg = cfg or PreprocessConfig()
    wav = _to_mono(wav)
    wav = _resample(wav, sample_rate, cfg.sample_rate)
    if cfg.highpass_hz > 0:
        wav = _highpass(wav, cfg.sample_rate, cfg.highpass_hz)
    if cfg.denoise:
        wav = _denoise(wav, cfg.sample_rate, cfg.denoise_prop)
    if cfg.peak_normalize:
        speech_mask = None
        if cfg.speech_normalize:
            speech_mask = speech_sample_mask(
                wav,
                cfg.sample_rate,
                thresh_db=cfg.speech_thresh_db,
                frame_ms=cfg.vad_frame_ms,
                hop_ms=cfg.vad_hop_ms,
            )
        wav = _peak_normalize(wav, cfg.target_peak, speech_mask)
    return wav.contiguous()


def load_waveform(
    path: PathLike,
    cfg: Optional[PreprocessConfig] = None,
    *,
    apply_preprocess: bool = True,
) -> torch.Tensor:
    """Load file; optionally run preprocessing chain."""
    cfg = cfg or PreprocessConfig()
    wav, sr = load_audio_file(path)
    if not apply_preprocess:
        wav = _to_mono(wav)
        wav = _resample(wav, sr, cfg.sample_rate)
        return wav.contiguous()
    return preprocess_waveform(wav, sr, cfg)


def truncate_waveform(
    wav: torch.Tensor,
    sample_rate: int,
    max_duration_sec: Optional[float],
) -> tuple[torch.Tensor, bool]:
    """Trim to max duration. None or <=0 = no limit."""
    if max_duration_sec is None or max_duration_sec <= 0:
        return wav, False
    max_samples = int(max_duration_sec * sample_rate)
    if wav.shape[0] <= max_samples:
        return wav, False
    return wav[:max_samples].contiguous(), True


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/audio_info.py
"""Audio file metadata helpers."""

from __future__ import annotations

from pathlib import Path
from typing import Union

PathLike = Union[str, Path]


def get_duration_sec(path: PathLike) -> float:
    """Return audio duration in seconds."""
    import soundfile as sf

    info = sf.info(str(path))
    return float(info.duration)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/cmudict.py
"""
CMU Pronouncing Dictionary loader.

Maps English words to ARPAbet phoneme sequences (same notation as SpeechOcean762).
Used when canonical phoneme sequence is needed for CTC forced alignment.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional


class CMUDict:
    """Load and query CMUdict for word -> phoneme lookup."""

  # ARPAbet phoneme pattern (e.g. AH0, T, SH)
    PHONE_RE = re.compile(r"^[A-Z]{1,2}\d?$")

    def __init__(self, dict_path: Optional[str] = None):
        self._lexicon: Dict[str, List[List[str]]] = {}
        if dict_path and Path(dict_path).exists():
            self._load_file(dict_path)
        else:
            self._load_nltk()

    def _load_file(self, path: str) -> None:
        """Parse standard CMUdict format: WORD  PHONE1 PHONE2 ..."""
        with open(path, encoding="latin-1") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith(";;;"):
                    continue
                if "(" in line:
                    # Alternate pronunciation: WORD(2)
                    word, rest = line.split("(", 1)
                    rest = rest.rstrip(")")
                    phones = rest.split()[1:]
                else:
                    parts = line.split()
                    word, phones = parts[0], parts[1:]
                key = word.lower()
                self._lexicon.setdefault(key, []).append(phones)

    def _load_nltk(self) -> None:
        """Fallback: download CMUdict via NLTK."""
        try:
            import nltk

            try:
                nltk.data.find("corpora/cmudict")
            except LookupError:
                nltk.download("cmudict", quiet=True)
            from nltk.corpus import cmudict

            for word, phones_list in cmudict.dict().items():
                self._lexicon[word.lower()] = [list(p) for p in phones_list]
        except Exception as exc:
            raise RuntimeError(
                "CMUdict not found. Provide data/cmudict/cmudict.dict or install nltk."
            ) from exc

    def lookup(self, word: str) -> Optional[List[str]]:
        """Return first pronunciation for word, or None."""
        variants = self._lexicon.get(word.lower().strip())
        return variants[0] if variants else None

    def text_to_phonemes(self, text: str) -> List[str]:
        """Convert whitespace-separated transcript to flat phoneme list."""
        phones: List[str] = []
        for word in text.upper().split():
            word_clean = re.sub(r"[^A-Z']", "", word)
            if not word_clean:
                continue
            pron = self.lookup(word_clean)
            if pron:
                phones.extend(pron)
        return phones

    def words_to_phoneme_groups(self, text: str) -> List[dict]:
        """
        Return per-word phoneme groups for graph edge construction.

        Each item: {"word": str, "phones": List[str]}
        """
        groups = []
        for word in text.upper().split():
            word_clean = re.sub(r"[^A-Z']", "", word)
            if not word_clean:
                continue
            pron = self.lookup(word_clean)
            if pron:
                groups.append({"word": word_clean, "phones": pron})
        return groups


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/silence_split.py
"""Split audio into speech segments at silence boundaries."""

from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import torch

from data.audio_preprocess import PreprocessConfig, frame_rms, load_waveform, save_audio_file, trim_silence_edges

PathLike = Union[str, Path]


@dataclass
class SilenceSplitConfig:
    min_silence_sec: float = 0.45
    silence_thresh_db: float = -35.0
    min_segment_sec: float = 0.4
    max_segment_sec: Optional[float] = None
    padding_sec: float = 0.08
    frame_ms: float = 25.0
    hop_ms: float = 10.0
    trim_edges: bool = True
    trim_min_sec: float = 0.05

    @classmethod
    def from_dict(cls, cfg: Optional[dict]) -> "SilenceSplitConfig":
        if not cfg:
            return cls()
        fields = cls.__dataclass_fields__
        return cls(**{k: v for k, v in cfg.items() if k in fields})


def split_waveform(
    wav: torch.Tensor,
    sample_rate: int,
    cfg: Optional[SilenceSplitConfig] = None,
) -> List[Dict[str, float]]:
    """Return [{start_sec, end_sec}, ...] speech segments."""
    cfg = cfg or SilenceSplitConfig()
    if wav.numel() == 0:
        return []

    rms = frame_rms(wav, sample_rate, cfg.frame_ms, cfg.hop_ms)
    ref = rms.max().clamp(min=1e-8)
    thresh = ref * (10 ** (cfg.silence_thresh_db / 20.0))
    speech = rms >= thresh

    hop_sec = cfg.hop_ms / 1000.0
    min_silence_frames = max(1, int(cfg.min_silence_sec / hop_sec))
    min_seg_samples = int(cfg.min_segment_sec * sample_rate)
    pad_samples = int(cfg.padding_sec * sample_rate)
    max_samples = int(cfg.max_segment_sec * sample_rate) if cfg.max_segment_sec else None

    boundaries = [0]
    n = len(speech)
    i = 0
    while i < n:
        if not speech[i]:
            j = i
            while j < n and not speech[j]:
                j += 1
            if j - i >= min_silence_frames:
                split_sample = int((i + (j - i) // 2) * hop_sec * sample_rate)
                if split_sample > boundaries[-1] + min_seg_samples:
                    boundaries.append(split_sample)
            i = j
        else:
            i += 1
    boundaries.append(wav.shape[0])

    segments: List[Dict[str, float]] = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        s = max(0, start - pad_samples)
        e = min(wav.shape[0], end + pad_samples)
        if e - s < min_seg_samples:
            continue
        if max_samples and e - s > max_samples:
            while e - s > max_samples:
                segments.append({
                    "start_sec": round(s / sample_rate, 3),
                    "end_sec": round((s + max_samples) / sample_rate, 3),
                })
                s += max_samples
        segments.append({
            "start_sec": round(s / sample_rate, 3),
            "end_sec": round(e / sample_rate, 3),
        })

    if not segments:
        segments.append({"start_sec": 0.0, "end_sec": round(wav.shape[0] / sample_rate, 3)})
    return segments


def split_active_regions(
    wav: torch.Tensor,
    sample_rate: int,
    cfg: Optional[SilenceSplitConfig] = None,
) -> List[Dict[str, float]]:
    """Split diarized tracks (zero-padded between turns) on amplitude gaps."""
    cfg = cfg or SilenceSplitConfig()
    if wav.numel() == 0:
        return []

    mono = wav if wav.dim() == 1 else wav.mean(0)
    peak = float(mono.abs().max())
    if peak < 1e-6:
        return [{"start_sec": 0.0, "end_sec": round(mono.shape[0] / sample_rate, 3)}]

    thresh = peak * (10 ** (cfg.silence_thresh_db / 20.0))
    active = mono.abs() >= thresh
    min_seg = int(cfg.min_segment_sec * sample_rate)
    pad = int(cfg.padding_sec * sample_rate)

    segments: List[Dict[str, float]] = []
    n = mono.shape[0]
    i = 0
    while i < n:
        while i < n and not active[i]:
            i += 1
        if i >= n:
            break
        start = i
        while i < n and active[i]:
            i += 1
        end = i
        if end - start < min_seg:
            continue
        s = max(0, start - pad)
        e = min(n, end + pad)
        segments.append({
            "start_sec": round(s / sample_rate, 3),
            "end_sec": round(e / sample_rate, 3),
        })

    if not segments:
        segments.append({"start_sec": 0.0, "end_sec": round(n / sample_rate, 3)})
    return segments


def _is_sparse_track(wav: torch.Tensor) -> bool:
    """True when many samples are near-zero (typical after 2-speaker diarization)."""
    mono = wav if wav.dim() == 1 else wav.mean(0)
    if mono.numel() == 0:
        return False
    return float((mono.abs() < 1e-5).float().mean()) > 0.2


def export_segments(
    wav: torch.Tensor,
    sample_rate: int,
    segments: List[Dict[str, float]],
    output_dir: PathLike,
    prefix: str = "sent",
    *,
    split_cfg: Optional[SilenceSplitConfig] = None,
) -> List[Dict[str, Any]]:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cfg = split_cfg or SilenceSplitConfig()
    min_samples = int(cfg.min_segment_sec * sample_rate)
    trim_min_samples = int(cfg.trim_min_sec * sample_rate)
    exported: List[Dict[str, Any]] = []

    for seg in segments:
        s = int(seg["start_sec"] * sample_rate)
        e = int(seg["end_sec"] * sample_rate)
        clip = wav[s:e].contiguous()

        trim_start = trim_end = 0
        if cfg.trim_edges:
            clip, trim_start, trim_end = trim_silence_edges(
                clip,
                sample_rate,
                thresh_db=cfg.silence_thresh_db,
                frame_ms=cfg.frame_ms,
                hop_ms=cfg.hop_ms,
                min_samples=trim_min_samples,
            )

        if clip.numel() < min_samples:
            continue

        idx = len(exported)
        path = out / f"{prefix}_{idx:03d}.wav"
        save_audio_file(path, clip.unsqueeze(0), sample_rate)
        duration = round(clip.shape[0] / sample_rate, 3)
        exported.append({
            "index": idx,
            "start_sec": round(seg["start_sec"] + trim_start / sample_rate, 3),
            "end_sec": round(seg["end_sec"] - trim_end / sample_rate, 3),
            "duration_sec": duration,
            "path": str(path),
            "preprocessed": True,
        })

    if not exported and wav.numel() > 0:
        mono = wav if wav.dim() == 1 else wav.mean(0)
        min_samples = int(cfg.min_segment_sec * sample_rate)
        if mono.numel() >= min_samples:
            path = out / f"{prefix}_000.wav"
            save_audio_file(path, mono.unsqueeze(0), sample_rate)
            exported.append({
                "index": 0,
                "start_sec": 0.0,
                "end_sec": round(mono.shape[0] / sample_rate, 3),
                "duration_sec": round(mono.shape[0] / sample_rate, 3),
                "path": str(path),
                "preprocessed": True,
            })
    return exported


def export_diarization_clips(
    source_audio: PathLike,
    diarize_segments: list,
    output_dir: PathLike,
    speaker_key: str,
    preprocess: Optional[PreprocessConfig] = None,
    *,
    merge_gap_sec: float = 0.35,
    min_duration_sec: float = 0.2,
    prefix: str = "sent",
) -> List[Dict[str, Any]]:
    """Cut clips from original audio using ECAPA diarization time spans."""
    label_map = {
        "A": "Speaker A",
        "B": "Speaker B",
        "TEACHER": "Teacher",
        "T": "Teacher",
        "STUDENT": "Student",
        "S": "Student",
    }
    label = label_map.get(speaker_key.upper(), speaker_key)
    spans: List[tuple[float, float]] = []
    for s in diarize_segments:
        sp = getattr(s, "speaker", None)
        if sp is None and isinstance(s, dict):
            sp = s.get("speaker")
        if sp != label:
            continue
        start = float(getattr(s, "start", 0) if not isinstance(s, dict) else s.get("start", 0))
        end = float(getattr(s, "end", 0) if not isinstance(s, dict) else s.get("end", 0))
        if end > start:
            spans.append((start, end))
    spans.sort()
    merged: List[tuple[float, float]] = []
    for start, end in spans:
        if end <= start:
            continue
        if merged and start - merged[-1][1] <= merge_gap_sec:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    preprocess = preprocess or PreprocessConfig()
    track_preprocess = replace(preprocess, denoise=False)
    wav = load_waveform(source_audio, track_preprocess)
    sr = track_preprocess.sample_rate
    mono = wav if wav.dim() == 1 else wav.mean(0)

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    min_samples = int(min_duration_sec * sr)
    exported: List[Dict[str, Any]] = []

    for start, end in merged:
        s = max(0, int(start * sr))
        e = min(mono.shape[0], int(end * sr))
        clip = mono[s:e].contiguous()
        if clip.numel() < min_samples:
            continue
        idx = len(exported)
        path = out / f"{prefix}_{idx:03d}.wav"
        save_audio_file(path, clip.unsqueeze(0), sr)
        exported.append({
            "index": idx,
            "start_sec": round(s / sr, 3),
            "end_sec": round(e / sr, 3),
            "duration_sec": round(clip.shape[0] / sr, 3),
            "path": str(path),
            "preprocessed": True,
        })
    return exported


def split_audio_file(
    audio_path: PathLike,
    output_dir: PathLike,
    preprocess: Optional[PreprocessConfig] = None,
    split_cfg: Optional[dict] = None,
    prefix: str = "sent",
) -> List[Dict[str, Any]]:
    preprocess = preprocess or PreprocessConfig()
    cfg = SilenceSplitConfig.from_dict(split_cfg)
    wav = load_waveform(audio_path, preprocess)
    sr = preprocess.sample_rate
    mono = wav if wav.dim() == 1 else wav.mean(0)
    if _is_sparse_track(mono):
        segments = split_active_regions(mono, sr, cfg)
        return export_segments(mono, sr, segments, output_dir, prefix, split_cfg=cfg)
    segments = split_waveform(wav, sr, cfg)
    return export_segments(wav, sr, segments, output_dir, prefix, split_cfg=cfg)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/__init__.py
from .wavlm_encoder import WavLMEncoder
from .transformer_encoder import TaskTransformerEncoder
from .ctc_aligner import CTCAligner
from .phoneme_graph import PhonemeGraphNetwork
from .multitask_heads import MultiTaskHeads
from .pronunciation_scorer import PronunciationScorer
from .pronunciation_model import PronunciationAssessmentModel

__all__ = [
    "WavLMEncoder", "TaskTransformerEncoder", "CTCAligner",
    "PhonemeGraphNetwork", "MultiTaskHeads", "PronunciationScorer",
    "PronunciationAssessmentModel",
]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/checkpoint_utils.py
"""Resolve and load transformer checkpoints from transformer_models/."""

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional, Union

import torch
import torch.nn as nn

PathLike = Union[str, Path]

DEFAULT_MODEL_DIR = Path("transformer_models")
PRONUNCIATION_NAMES = ("pronunciation.pt", "best_model.pt")
MDD_NAMES = ("l2_mdd.pt", "best_model.pt")


def _first_existing(base: Path, names: tuple[str, ...]) -> Optional[Path]:
    for name in names:
        p = base / name
        if p.is_file():
            return p
    return None


def _download_pronunciation_from_hub(config: Dict[str, Any]) -> Optional[Path]:
    paths = config.get("paths", {})
    repo_id = paths.get("pronunciation_hf_repo")
    if not repo_id:
        return None

    filename = paths.get("pronunciation_hf_filename", "pronunciation.pt")
    model_dir = Path(paths.get("transformer_models_dir", DEFAULT_MODEL_DIR))
    model_dir.mkdir(parents=True, exist_ok=True)
    target = model_dir / filename

    from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(model_dir))
    return target if target.is_file() else None


def resolve_checkpoint(
    config: Dict[str, Any],
    *,
    model: str,
    explicit: Optional[PathLike] = None,
) -> Path:
    """model: pronunciation | l2_mdd"""
    if explicit:
        p = Path(explicit)
        if p.is_file():
            return p
        raise FileNotFoundError(f"Checkpoint not found: {p}")

    paths = config.get("paths", {})
    key = "pronunciation_checkpoint" if model == "pronunciation" else "l2_mdd_checkpoint"
    configured = paths.get(key)
    if configured and Path(configured).is_file():
        return Path(configured)

    model_dir = Path(paths.get("transformer_models_dir", DEFAULT_MODEL_DIR))
    names = PRONUNCIATION_NAMES if model == "pronunciation" else MDD_NAMES
    found = _first_existing(model_dir, names)
    if found:
        return found

    legacy_dir = Path(
        paths.get("checkpoint_dir", "checkpoints" if model == "pronunciation" else "checkpoints/l2_mdd")
    )
    legacy = legacy_dir / "best_model.pt"
    if legacy.is_file():
        return legacy

    if model == "pronunciation":
        downloaded = _download_pronunciation_from_hub(config)
        if downloaded:
            return downloaded
        found = _first_existing(model_dir, names)
        if found:
            return found

    expected = ", ".join(names)
    raise FileNotFoundError(
        f"No {model} checkpoint. Place weights in {model_dir}/ ({expected}) "
        f"or set paths.{key} in config."
    )


def load_state_dict(path: PathLike, device: Union[str, torch.device] = "cpu") -> dict:
    st = torch.load(path, map_location=device, weights_only=False)
    if isinstance(st, dict) and "model_state_dict" in st:
        return st["model_state_dict"]
    return st


def load_model_weights(
    model: nn.Module,
    config: Dict[str, Any],
    *,
    model_kind: str,
    explicit: Optional[PathLike] = None,
    device: Union[str, torch.device] = "cpu",
    strict: bool = True,
) -> Path:
    path = resolve_checkpoint(config, model=model_kind, explicit=explicit)
    model.load_state_dict(load_state_dict(path, device), strict=strict)
    return path


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/ctc_aligner.py
"""
CTC Forced Alignment for phoneme-level timestamp extraction.

Step 3 of pipeline:
  1. Project WavLM frame features -> CTC log-probs over phoneme vocabulary.
  2. Run torchaudio.functional.forced_align with canonical phoneme sequence.
  3. Convert alignment spans -> pooled phoneme node features for Graph Attention.

Key conversion (alignment -> node features):
  For each phoneme p_i aligned to frame range [t_start, t_end]:
      node_feature_i = mean(frame_hidden[t_start:t_end+1])
  This pools acoustic+linguistic context from the task Transformer output
  into one vector per phoneme, which becomes the initial GAT node embedding.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

# Lazy import: torchaudio may fail on some Windows/Python combos
torchaudio = None
forced_align = None


def _ensure_torchaudio():
    global torchaudio, forced_align
    if torchaudio is not None:
        return True
    try:
        import torchaudio as _ta
        from torchaudio.functional import forced_align as _fa

        torchaudio = _ta
        forced_align = _fa
        return True
    except (ImportError, OSError):
        return False


# espeak-style phoneme set subset (ARPAbet compatible with SpeechOcean762)
DEFAULT_PHONEME_VOCAB = [
    "<pad>", "<unk>", "|",  # blank, unknown, word boundary
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2",
    "AO0", "AO1", "AO2", "AW0", "AW1", "AW2", "AY0", "AY1", "AY2",
    "B", "CH", "D", "DH", "EH0", "EH1", "EH2", "ER0", "ER1", "ER2",
    "EY0", "EY1", "EY2", "F", "G", "HH", "IH0", "IH1", "IH2",
    "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH",
    "T", "TH", "UH0", "UH1", "UH2", "UW0", "UW1", "UW2",
    "V", "W", "Y", "Z", "ZH",
]


@dataclass
class PhonemeAlignment:
    """Single phoneme alignment result."""

    phoneme: str
    token_id: int
    start_frame: int
    end_frame: int
    confidence: float


class CTCAligner(nn.Module):
    """
    CTC head + forced alignment to map phoneme sequence -> frame spans.

    Can optionally use a pretrained torchaudio CTC bundle for alignment-only
    mode; default trains a lightweight linear CTC head on WavLM features.
    """

    def __init__(
        self,
        input_dim: int,
        phoneme_vocab: Optional[List[str]] = None,
        use_pretrained_bundle: bool = False,
        blank_id: int = 0,
    ):
        super().__init__()
        self.phoneme_vocab = phoneme_vocab or DEFAULT_PHONEME_VOCAB
        self.token2id = {p: i for i, p in enumerate(self.phoneme_vocab)}
        self.blank_id = blank_id
        self.num_tokens = len(self.phoneme_vocab)

        self.ctc_proj = nn.Linear(input_dim, self.num_tokens)
        self.use_pretrained_bundle = use_pretrained_bundle
        self._bundle = None

        if use_pretrained_bundle and _ensure_torchaudio():
            try:
                # Phoneme ASR model (espeak phonemes) when available
                self._bundle = torchaudio.pipelines.MMS_FA
            except AttributeError:
                self._bundle = None

    def phonemes_to_ids(self, phonemes: List[str]) -> torch.Tensor:
        """Map ARPAbet phoneme strings to token IDs."""
        ids = []
        for p in phonemes:
            ids.append(self.token2id.get(p, self.token2id["<unk>"]))
        return torch.tensor(ids, dtype=torch.long)

    def forward_ctc_logits(self, frame_features: torch.Tensor) -> torch.Tensor:
        """(B, T, D) -> (T, B, C) log-probs for torch.nn.functional.ctc_loss."""
        logits = self.ctc_proj(frame_features)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.transpose(0, 1)

    def _align_log_probs(self, frame_features: torch.Tensor) -> torch.Tensor:
        """(T, D) -> (1, T, C) log-probs for torchaudio forced_align (batch-first)."""
        logits = self.ctc_proj(frame_features.unsqueeze(0))
        return F.log_softmax(logits, dim=-1)

    def _run_forced_align(
        self,
        log_probs: torch.Tensor,
        targets: torch.Tensor,
        input_lengths: torch.Tensor,
        target_lengths: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """forced_align wants (B, T, C) with B=1; GPU kernel errors on wrong layout."""
        assert log_probs.dim() == 3 and log_probs.shape[0] == 1
        try:
            return forced_align(
                log_probs.float(),
                targets.unsqueeze(0),
                input_lengths,
                target_lengths,
                blank=self.blank_id,
            )
        except RuntimeError:
            return forced_align(
                log_probs.float().cpu(),
                targets.unsqueeze(0).cpu(),
                input_lengths.cpu(),
                target_lengths.cpu(),
                blank=self.blank_id,
            )

    def align_utterance(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
        frame_lengths: Optional[int] = None,
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        """
        Force-align one utterance.

        Args:
            frame_features: (T, D) single utterance frame features.
            phonemes: canonical ARPAbet phoneme list from transcript/CMUdict.
            frame_lengths: number of valid frames T.

        Returns:
            alignments: list of PhonemeAlignment with frame spans.
            node_features: (num_phonemes, D) pooled features for GAT nodes.
        """
        if frame_lengths is None:
            frame_lengths = frame_features.shape[0]

        T, D = frame_features.shape
        if len(phonemes) == 0:
            return [], frame_features.new_zeros(0, D)

        targets = self.phonemes_to_ids(phonemes).to(frame_features.device)

        if not _ensure_torchaudio() or forced_align is None:
            return self._uniform_align(frame_features, phonemes)

        log_probs = self._align_log_probs(frame_features)  # (1, T, C)
        input_lengths = torch.tensor([frame_lengths], device=frame_features.device)
        target_lengths = torch.tensor([len(targets)], device=frame_features.device)

        aligned_tokens, _align_scores = self._run_forced_align(
            log_probs, targets, input_lengths, target_lengths
        )
        if aligned_tokens.dim() > 1:
            aligned = aligned_tokens[0, :frame_lengths]
        else:
            aligned = aligned_tokens[:frame_lengths]

        spans = self._tokens_to_spans(aligned, targets, phonemes)
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def _tokens_to_spans(
        self,
        aligned: torch.Tensor,
        targets: torch.Tensor,
        phonemes: List[str],
    ) -> List[PhonemeAlignment]:
        """
        Parse per-frame CTC alignment into phoneme start/end spans.

        forced_align output assigns each frame to a target token index or blank.
        Consecutive frames with the same non-blank token index form one span.
        """
        spans: List[PhonemeAlignment] = []
        target_list = targets.tolist()
        i = 0
        while i < len(phonemes):
            token_id = target_list[i]
            # find frames assigned to this target position
            mask = aligned == i  # forced_align uses target position indices
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0]
                start_f, end_f = int(idx[0]), int(idx[-1])
                conf = float(mask.float().mean())
            else:
                # phoneme got no frames — interpolate
                start_f = end_f = 0
                conf = 0.0
            spans.append(
                PhonemeAlignment(
                    phoneme=phonemes[i],
                    token_id=token_id,
                    start_frame=start_f,
                    end_frame=end_f,
                    confidence=conf,
                )
            )
            i += 1
        return spans

    def _pool_node_features(
        self,
        frame_features: torch.Tensor,
        spans: List[PhonemeAlignment],
    ) -> torch.Tensor:
        """
        Pool frame hidden states over [t_start, t_end] for each phoneme.

        This is the critical bridge from CTC alignment to Graph Attention:
        each phoneme node receives a fixed-size embedding regardless of
        how many frames it spans.
        """
        nodes = []
        T = frame_features.shape[0]
        for span in spans:
            s = max(0, span.start_frame)
            e = min(T - 1, span.end_frame)
            if s <= e:
                pooled = frame_features[s : e + 1].mean(dim=0)
            else:
                pooled = frame_features.mean(dim=0)
            nodes.append(pooled)
        return torch.stack(nodes, dim=0) if nodes else frame_features.new_zeros(0, frame_features.shape[-1])

    def _uniform_align(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        """Fallback equal-split alignment when forced_align is unavailable."""
        T, D = frame_features.shape
        n = max(len(phonemes), 1)
        chunk = T // n
        spans = []
        for i, ph in enumerate(phonemes):
            s = i * chunk
            e = min(T - 1, (i + 1) * chunk - 1) if i < n - 1 else T - 1
            tid = self.token2id.get(ph, self.token2id["<unk>"])
            spans.append(
                PhonemeAlignment(ph, tid, s, e, 1.0)
            )
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def batch_align(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
    ) -> List[Tuple[List[PhonemeAlignment], torch.Tensor]]:
        """Align a batch; returns per-utterance (spans, node_features)."""
        results = []
        B = frame_features.shape[0]
        for b in range(B):
            T_b = int(frame_lengths[b].item())
            spans, nodes = self.align_utterance(
                frame_features[b, :T_b],
                phoneme_lists[b],
                T_b,
            )
            results.append((spans, nodes))
        return results

    def ctc_loss(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
    ) -> torch.Tensor:
        """Auxiliary CTC loss for training the alignment head."""
        log_probs = self.forward_ctc_logits(frame_features)
        targets_list = []
        target_lengths = []
        for phs in phoneme_lists:
            t = self.phonemes_to_ids(phs)
            targets_list.append(t)
            target_lengths.append(len(t))
        targets = torch.cat(targets_list).to(frame_features.device)
        target_lengths_t = torch.tensor(target_lengths, device=frame_features.device)
        loss = F.ctc_loss(
            log_probs,
            targets,
            frame_lengths,
            target_lengths_t,
            blank=self.blank_id,
            zero_infinity=True,
        )
        return loss


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/multitask_heads.py
"""
Multi-task regression heads (GOPT-inspired).

Step 5: parallel heads for pronunciation aspects at 3 granularities.

Architecture mirrors GOPT (ICASSP 2022):
  - Each head: LayerNorm -> Linear(hidden, 1) regression.
  - Utterance heads: applied on utterance-level pooled representation.
  - Word heads: applied on mean-pool of phoneme nodes per word.
  - Phoneme heads: applied on each GAT output node.

Aspects mapped to SpeechOcean762 labels:
  Utterance: accuracy, fluency, completeness, prosodic, total
  Word:      accuracy, stress, total
  Phoneme:   accuracy (phones-accuracy)
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn


class RegressionHead(nn.Module):
    """Single-aspect regression head with layer norm (GOPT style)."""

    def __init__(self, input_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
            nn.Linear(input_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


class MultiTaskHeads(nn.Module):
    """Collection of aspect-specific heads at phoneme/word/utterance levels."""

    def __init__(
        self,
        input_dim: int,
        utterance_aspects: Optional[List[str]] = None,
        word_aspects: Optional[List[str]] = None,
        phoneme_aspects: Optional[List[str]] = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.utterance_aspects = utterance_aspects or [
            "accuracy", "fluency", "completeness", "prosodic", "total"
        ]
        self.word_aspects = word_aspects or ["accuracy", "stress", "total"]
        self.phoneme_aspects = phoneme_aspects or ["accuracy"]
        self.input_dim = input_dim

        self.utt_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.utterance_aspects}
        )
        self.word_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.word_aspects}
        )
        self.phoneme_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.phoneme_aspects}
        )

    def pool_utterance(self, phoneme_embeddings: torch.Tensor) -> torch.Tensor:
        """Mean-pool all phoneme nodes -> utterance representation."""
        if phoneme_embeddings.shape[0] == 0:
            return phoneme_embeddings.new_zeros(self.input_dim)
        return phoneme_embeddings.mean(dim=0)

    def pool_words(
        self,
        phoneme_embeddings: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        """Mean-pool phoneme nodes per word -> (num_words, D)."""
        word_embs = []
        for start, end in word_phone_ranges:
            if start < end:
                word_embs.append(phoneme_embeddings[start:end].mean(dim=0))
            else:
                word_embs.append(phoneme_embeddings.new_zeros(self.input_dim))
        return torch.stack(word_embs, dim=0) if word_embs else phoneme_embeddings.new_zeros(0, self.input_dim)

    def forward_single(
        self,
        phoneme_embeddings: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> Dict[str, torch.Tensor]:
        """
        Predict all aspects for one utterance.

        Returns dict with keys:
          utterance_{aspect}, word_{aspect} (W,), phoneme_{aspect} (N,)
        """
        utt_repr = self.pool_utterance(phoneme_embeddings)
        word_repr = self.pool_words(phoneme_embeddings, word_phone_ranges)

        out: Dict[str, torch.Tensor] = {}
        for aspect, head in self.utt_heads.items():
            out[f"utterance_{aspect}"] = head(utt_repr.unsqueeze(0)).squeeze(0)
        for aspect, head in self.word_heads.items():
            if word_repr.shape[0] > 0:
                out[f"word_{aspect}"] = head(word_repr)
            else:
                out[f"word_{aspect}"] = phoneme_embeddings.new_zeros(0)
        for aspect, head in self.phoneme_heads.items():
            if phoneme_embeddings.shape[0] > 0:
                out[f"phoneme_{aspect}"] = head(phoneme_embeddings)
            else:
                out[f"phoneme_{aspect}"] = phoneme_embeddings.new_zeros(0)
        return out

    def forward_batch(
        self,
        phoneme_embeddings_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[Dict[str, torch.Tensor]]:
        return [
            self.forward_single(pe, wr)
            for pe, wr in zip(phoneme_embeddings_list, word_phone_ranges_list)
        ]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/phoneme_graph.py
"""
Graph Attention Network over phoneme nodes.

Step 4: build per-utterance phoneme graph and apply GATv2Conv layers.

Graph construction per utterance:
  - Nodes: one per phoneme instance, initial feature = pooled frame embedding
    from CTC alignment (see ctc_aligner._pool_node_features).
  - Edges:
      * Sequential: (i, i+1) bidirectional — captures coarticulation context.
      * Same-word: all phoneme pairs within one word — captures lexical stress.
      * Same-syllable (optional): pairs within syllable group.

After GAT message passing, each node has context-aware embedding used by
multi-task regression heads at phoneme level; word/utterance levels pool
these node embeddings.
"""

from __future__ import annotations

from typing import List, Optional, Tuple

import torch
import torch.nn as nn

try:
    from torch_geometric.nn import GATv2Conv
except ImportError:
    GATv2Conv = None


class PhonemeGraphNetwork(nn.Module):
    """GATv2-based phoneme graph encoder."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
        edge_sequential: bool = True,
        edge_same_word: bool = True,
        edge_same_syllable: bool = False,
    ):
        super().__init__()
        if GATv2Conv is None:
            raise ImportError("torch_geometric is required for PhonemeGraphNetwork")

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.edge_sequential = edge_sequential
        self.edge_same_word = edge_same_word
        self.edge_same_syllable = edge_same_syllable

        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            in_ch = hidden_dim
            out_ch = hidden_dim // num_heads
            self.gat_layers.append(
                GATv2Conv(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    heads=num_heads,
                    dropout=dropout,
                    concat=True,
                )
            )
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim

    @staticmethod
    def build_edge_index(
        num_phonemes: int,
        word_phone_ranges: List[Tuple[int, int]],
        sequential: bool = True,
        same_word: bool = True,
        same_syllable: bool = False,
    ) -> torch.Tensor:
        """
        Build COO edge_index (2, E) for one utterance.

        Args:
            num_phonemes: total phoneme count.
            word_phone_ranges: list of (start, end) exclusive indices per word.
        """
        edges = set()

        if sequential:
            for i in range(num_phonemes - 1):
                edges.add((i, i + 1))
                edges.add((i + 1, i))

        if same_word:
            for start, end in word_phone_ranges:
                for i in range(start, end):
                    for j in range(start, end):
                        if i != j:
                            edges.add((i, j))

        if same_syllable:
            # simple heuristic: split each word's phones in half
            for start, end in word_phone_ranges:
                mid = (start + end) // 2
                for i in range(start, mid):
                    for j in range(start, mid):
                        if i != j:
                            edges.add((i, j))
                for i in range(mid, end):
                    for j in range(mid, end):
                        if i != j:
                            edges.add((i, j))

        if not edges:
            # self-loop fallback for single phoneme
            edges.add((0, 0))

        src, dst = zip(*edges)
        return torch.tensor([src, dst], dtype=torch.long)

    def forward_single(
        self,
        node_features: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        """
        Args:
            node_features: (N, D) pooled phoneme embeddings from CTC align step.
            word_phone_ranges: phoneme index ranges per word.

        Returns:
            (N, hidden_dim) context-enriched phoneme embeddings.
        """
        x = self.input_proj(node_features)
        edge_index = self.build_edge_index(
            node_features.shape[0],
            word_phone_ranges,
            self.edge_sequential,
            self.edge_same_word,
            self.edge_same_syllable,
        ).to(node_features.device)

        for gat in self.gat_layers:
            x = gat(x, edge_index)
            x = self.dropout(torch.relu(x))
        return self.norm(x)

    def forward_batch(
        self,
        node_features_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[torch.Tensor]:
        """Process variable-size graphs per utterance."""
        outputs = []
        for nodes, ranges in zip(node_features_list, word_phone_ranges_list):
            if nodes.shape[0] == 0:
                outputs.append(nodes.new_zeros(0, self.output_dim))
            else:
                outputs.append(self.forward_single(nodes, ranges))
        return outputs


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/pronunciation_model.py
"""
End-to-end Pronunciation Assessment model.

Full pipeline:
  waveform -> WavLM -> TaskTransformer -> CTC align -> GAT graph -> multi-task heads
"""

from __future__ import annotations

from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn

from .wavlm_encoder import WavLMEncoder
from .transformer_encoder import TaskTransformerEncoder
from .ctc_aligner import CTCAligner
from .phoneme_graph import PhonemeGraphNetwork
from .multitask_heads import MultiTaskHeads


class PronunciationAssessmentModel(nn.Module):
    """Full pronunciation assessment pipeline."""

    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        wavlm_cfg = config.get("wavlm", {})
        trans_cfg = config.get("transformer", {})
        ctc_cfg = config.get("ctc_align", {})
        graph_cfg = config.get("phoneme_graph", {})
        mt_cfg = config.get("multitask", {})

        self.wavlm = WavLMEncoder(
            model_name=wavlm_cfg.get("model_name", "microsoft/wavlm-large"),
            freeze=wavlm_cfg.get("freeze", True),
            use_lora=wavlm_cfg.get("use_lora", False),
            lora_r=wavlm_cfg.get("lora_r", 8),
            lora_alpha=wavlm_cfg.get("lora_alpha", 16),
            lora_dropout=wavlm_cfg.get("lora_dropout", 0.05),
            lora_target_modules=wavlm_cfg.get("lora_target_modules"),
        )
        d = self.wavlm.output_dim

        self.task_transformer = TaskTransformerEncoder(
            input_dim=d,
            num_layers=trans_cfg.get("num_layers", 3),
            num_heads=trans_cfg.get("num_heads", 8),
            ff_dim=trans_cfg.get("ff_dim", 3072),
            dropout=trans_cfg.get("dropout", 0.1),
            max_seq_len=trans_cfg.get("max_seq_len", 2000),
        )

        self.ctc_aligner = CTCAligner(
            input_dim=d,
            use_pretrained_bundle=not ctc_cfg.get("use_wavlm_ctc_head", True),
        )

        graph_hidden = graph_cfg.get("hidden_dim", 256)
        self.phoneme_graph = PhonemeGraphNetwork(
            input_dim=d,
            hidden_dim=graph_hidden,
            num_layers=graph_cfg.get("num_gat_layers", 2),
            num_heads=graph_cfg.get("num_heads", 4),
            dropout=graph_cfg.get("dropout", 0.1),
            edge_sequential=graph_cfg.get("edge_types", {}).get("sequential", True),
            edge_same_word=graph_cfg.get("edge_types", {}).get("same_word", True),
            edge_same_syllable=graph_cfg.get("edge_types", {}).get("same_syllable", False),
        )

        self.multitask_heads = MultiTaskHeads(
            input_dim=graph_hidden,
            utterance_aspects=mt_cfg.get("utterance_aspects"),
            word_aspects=mt_cfg.get("word_aspects"),
            phoneme_aspects=mt_cfg.get("phoneme_aspects"),
            dropout=mt_cfg.get("dropout", 0.1),
        )

        self.sample_rate = config.get("train", {}).get("dataset", {}).get("sample_rate", 16000)

    def _frame_lengths_from_wave(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        """Convert sample lengths to WavLM frame lengths using model conv math."""
        return self.wavlm.frame_lengths_from_samples(wav_lengths)

    def forward(
        self,
        waveforms: torch.Tensor,
        wav_lengths: torch.Tensor,
        phoneme_tokens: List[List[str]],
        word_phone_ranges: List[List[tuple]],
        return_alignments: bool = False,
    ) -> Dict[str, Any]:
        """
        Forward pass for a batch.

        Args:
            waveforms: (B, samples) padded.
            wav_lengths: (B,) actual sample counts.
            phoneme_tokens: list of phoneme strings per utterance.
            word_phone_ranges: word -> phoneme index ranges.

        Returns:
            dict with per-utterance predictions, optional alignments, ctc_loss.
        """
        # Step 1: WavLM features
        frame_feats = self.wavlm(waveforms, wav_lengths=wav_lengths)  # (B, T, D)
        frame_lengths = self._frame_lengths_from_wave(wav_lengths)

        # padding mask for transformer
        T = frame_feats.shape[1]
        pad_mask = torch.arange(T, device=waveforms.device).unsqueeze(0) >= frame_lengths.unsqueeze(1)

        # Step 2: task transformer
        frame_feats = self.task_transformer(frame_feats, src_key_padding_mask=pad_mask)

        # Step 3: CTC alignment -> phoneme node features
        align_results = self.ctc_aligner.batch_align(
            frame_feats, phoneme_tokens, frame_lengths
        )
        node_features_list = [nodes for _, nodes in align_results]
        alignments_list = [spans for spans, _ in align_results] if return_alignments else None

        # Step 4: graph attention
        graph_out = self.phoneme_graph.forward_batch(
            node_features_list, word_phone_ranges
        )

        # Step 5: multi-task heads
        predictions = self.multitask_heads.forward_batch(graph_out, word_phone_ranges)

        # auxiliary CTC loss
        ctc_loss = self.ctc_aligner.ctc_loss(frame_feats, phoneme_tokens, frame_lengths)

        out = {
            "predictions": predictions,
            "ctc_loss": ctc_loss,
            "graph_embeddings": graph_out,
        }
        if return_alignments:
            out["alignments"] = alignments_list
        return out

    @classmethod
    def from_config_path(cls, path: str) -> "PronunciationAssessmentModel":
        import yaml

        with open(path, encoding="utf-8") as f:
            config = yaml.safe_load(f)
        return cls(config)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/pronunciation_scorer.py
"""
Pronunciation score aggregation (0-10 scale).

Step 6: combine multi-task head outputs into interpretable final scores.
"""

from __future__ import annotations

from typing import Dict, List, Optional

import torch


class PronunciationScorer:
    """
    Aggregate multi-granularity predictions into final 0-10 scores.

    Training targets are normalized to 0-2 (GOPT convention); this class
    denormalizes back to SpeechOcean762's 0-10 scale for reporting.
    """

    def __init__(
        self,
        score_scale: float = 5.0,
        weights: Optional[Dict[str, float]] = None,
        phoneme_low_threshold: float = 1.2,
    ):
        self.score_scale = score_scale
        self.weights = weights or {
            "utterance_total": 0.4,
            "word_total": 0.3,
            "phoneme_accuracy": 0.3,
        }
        self.phoneme_low_threshold = phoneme_low_threshold

    def to_display_scale(self, score: float) -> float:
        """Map 0-2 normalized score -> 0-10 display scale."""
        return min(10.0, max(0.0, score * self.score_scale))

    def aggregate_utterance(self, predictions: Dict[str, torch.Tensor]) -> Dict[str, float]:
        """Build utterance-level score dict on 0-10 scale."""
        result = {}
        for key, val in predictions.items():
            if key.startswith("utterance_"):
                aspect = key.replace("utterance_", "")
                if isinstance(val, torch.Tensor):
                    val = float(val.detach().cpu().item())
                result[aspect] = self.to_display_scale(val)
        return result

    def final_score(self, predictions: Dict[str, torch.Tensor]) -> float:
        """
        Weighted combination of total/accuracy signals -> single 0-10 score.
        """
        parts = []
        w_sum = 0.0

        if "utterance_total" in predictions:
            v = predictions["utterance_total"]
            v = float(v.detach().cpu().item()) if isinstance(v, torch.Tensor) else v
            parts.append(self.weights["utterance_total"] * self.to_display_scale(v))
            w_sum += self.weights["utterance_total"]

        if "word_total" in predictions:
            wt = predictions["word_total"]
            if isinstance(wt, torch.Tensor) and wt.numel() > 0:
                v = float(wt.mean().detach().cpu().item())
                parts.append(self.weights["word_total"] * self.to_display_scale(v))
                w_sum += self.weights["word_total"]

        if "phoneme_accuracy" in predictions:
            pa = predictions["phoneme_accuracy"]
            if isinstance(pa, torch.Tensor) and pa.numel() > 0:
                v = float(pa.mean().detach().cpu().item())
                parts.append(self.weights["phoneme_accuracy"] * self.to_display_scale(v))
                w_sum += self.weights["phoneme_accuracy"]

        if w_sum == 0:
            return 0.0
        return sum(parts) / w_sum

    def find_errors(
        self,
        predictions: Dict[str, torch.Tensor],
        phoneme_tokens: List[str],
        word_texts: List[str],
        word_phone_ranges: List[tuple],
        threshold: Optional[float] = None,
    ) -> Dict[str, List[dict]]:
        """
        Identify low-scoring phonemes/words for LLM feedback.

        Returns:
            {"phonemes": [...], "words": [...]} with scores on 0-10 scale.
        """
        thr = threshold if threshold is not None else self.phoneme_low_threshold
        errors = {"phonemes": [], "words": []}

        pa = predictions.get("phoneme_accuracy")
        if pa is not None and isinstance(pa, torch.Tensor):
            for i, (tok, score) in enumerate(zip(phoneme_tokens, pa.tolist())):
                if score < thr:
                    errors["phonemes"].append(
                        {
                            "index": i,
                            "phoneme": tok,
                            "score": self.to_display_scale(score),
                        }
                    )

        wt = predictions.get("word_accuracy")
        if wt is not None and isinstance(wt, torch.Tensor):
            for i, (word, score) in enumerate(zip(word_texts, wt.tolist())):
                display = self.to_display_scale(score)
                if display < self.to_display_scale(thr):
                    errors["words"].append(
                        {"index": i, "word": word, "score": display}
                    )

        return errors


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/transformer_encoder.py
"""
Task-specific Transformer encoder stacked on WavLM features.

Step 2: refines frame-level representations for pronunciation assessment.
"""

from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn


class TaskTransformerEncoder(nn.Module):
    """
    Additional Transformer encoder layers (2-4) on top of WavLM hidden states.

    Uses pre-norm TransformerEncoderLayer for training stability.
    """

    def __init__(
        self,
        input_dim: int,
        num_layers: int = 3,
        num_heads: int = 8,
        ff_dim: int = 4096,
        dropout: float = 0.1,
        max_seq_len: int = 2000,
    ):
        super().__init__()
        if input_dim % num_heads != 0:
            raise ValueError(
                f"input_dim ({input_dim}) must be divisible by num_heads ({num_heads})"
            )
        self.input_proj = nn.Linear(input_dim, input_dim)
        self.pos_encoding = SinusoidalPositionalEncoding(input_dim, max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_dim = input_dim

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (B, T, D) frame features from WavLM.
            src_key_padding_mask: (B, T) True = padded frame (ignore).

        Returns:
            (B, T, D) refined frame features.
        """
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)


class SinusoidalPositionalEncoding(nn.Module):
    """Standard sinusoidal position encoding added to frame features."""

    def __init__(self, dim: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float) * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            self._grow_pe(seq_len)
        return x + self.pe[:, :seq_len, :].to(dtype=x.dtype, device=x.device)

    def _grow_pe(self, seq_len: int) -> None:
        dim = self.pe.size(-1)
        pe = torch.zeros(seq_len, dim, device=self.pe.device, dtype=self.pe.dtype)
        position = torch.arange(0, seq_len, dtype=torch.float, device=self.pe.device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float, device=self.pe.device)
            * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/wavlm_encoder.py
"""
WavLM Large feature extractor.

Step 1 of pipeline: waveform -> frame-level hidden states (B, T, D).
Supports frozen backbone or LoRA fine-tuning via PEFT.
"""

from __future__ import annotations

from contextlib import nullcontext
from typing import Optional

import torch
import torch.nn as nn
from transformers import WavLMModel


class WavLMEncoder(nn.Module):
    """
    Wraps `microsoft/wavlm-large` for pronunciation feature extraction.

    Output: frame-level representations at ~20ms stride (50 Hz for 16kHz audio
    with conv subsampling factor 320: 16000/320 = 50 frames/sec).
    """

    def __init__(
        self,
        model_name: str = "microsoft/wavlm-large",
        freeze: bool = True,
        use_lora: bool = False,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.05,
        lora_target_modules: Optional[list] = None,
    ):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(model_name)
        self.output_dim = self.wavlm.config.hidden_size  # 1024 for large

        if freeze and not use_lora:
            for p in self.wavlm.parameters():
                p.requires_grad = False
            self.wavlm.eval()

        self._frozen = freeze and not use_lora

        if use_lora:
            from peft import LoraConfig, get_peft_model

            target = lora_target_modules or ["q_proj", "v_proj"]
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=target,
                lora_dropout=lora_dropout,
                bias="none",
            )
            self.wavlm = get_peft_model(self.wavlm, lora_config)

    def forward(
        self,
        waveform: torch.Tensor,
        wav_lengths: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            waveform: (B, num_samples) float tensor, 16kHz mono.
            wav_lengths: (B,) actual sample counts before padding.
            attention_mask: optional (B, num_samples), 1=valid 0=pad.

        Returns:
            hidden_states: (B, T_frames, D) last hidden layer output.
        """
        if attention_mask is None:
            B, S = waveform.shape
            if wav_lengths is not None:
                attention_mask = (
                    torch.arange(S, device=waveform.device).unsqueeze(0)
                    < wav_lengths.unsqueeze(1)
                ).long()
            else:
                # padded batch: trailing zeros from pad_sequence
                attention_mask = (waveform.abs() > 1e-8).long()

        if self._frozen:
            self.wavlm.eval()

        ctx = torch.inference_mode if self._frozen else nullcontext
        with ctx():
            outputs = self.wavlm(
                input_values=waveform,
                attention_mask=attention_mask,
            )
        return outputs.last_hidden_state

    def frame_lengths_from_samples(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        """Exact WavLM frame counts from raw sample lengths (not samples//320)."""
        return self.wavlm._get_feat_extract_output_lengths(wav_lengths).long()

    def frame_rate(self, sample_rate: int = 16000) -> float:
        """Approximate frame rate (Hz) of WavLM output."""
        # WavLM conv feature extractor: total stride 320 samples
        return sample_rate / 320.0


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/__init__.py
"""Inference: pronunciation scoring (+ optional diarize)."""

from infer.pipeline import SpeakingPipeline
from infer.pronunciation import Predictor
from infer.transcribe import transcribe_audio

__all__ = ["Predictor", "SpeakingPipeline", "transcribe_audio"]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/device_utils.py
"""Resolve compute device for inference (GPU / CPU)."""

from __future__ import annotations

import os
from typing import Any, Dict, Optional

import torch


def resolve_device(
    config: Optional[Dict[str, Any]] = None,
    explicit: Optional[str] = None,
) -> str:
    """Prefer explicit arg → env → config → auto cuda/cpu."""
    if explicit:
        return _normalize(explicit)

    for key in ("DEVICE", "TORCH_DEVICE", "HF_DEVICE"):
        if os.getenv(key):
            return _normalize(os.getenv(key))

    cfg = config or {}
    inf = cfg.get("inference") or {}
    if inf.get("device"):
        return _normalize(inf["device"])

    proj = cfg.get("project") or {}
    if proj.get("device"):
        return _normalize(proj["device"])

    return "cuda" if torch.cuda.is_available() else "cpu"


def _normalize(device: str) -> str:
    device = device.strip().lower()
    if device in ("gpu", "cuda", "cuda:0"):
        return "cuda" if torch.cuda.is_available() else "cpu"
    if device.startswith("cuda") and not torch.cuda.is_available():
        return "cpu"
    return device


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/ensure_models.py
"""Download ECAPA + Silero at startup when not bundled in the repo (HF Spaces)."""

from __future__ import annotations

import sys
from pathlib import Path

from paths import ECAPA_DIR

ECAPA_WEIGHTS = ECAPA_DIR / "pretrained_models" / "spkrec-ecapa-voxceleb" / "hyperparams.yaml"
TORCH_HUB_DIR = ECAPA_DIR / "pretrained_models" / "torch_hub"


def _silero_ready() -> bool:
    if not TORCH_HUB_DIR.exists():
        return False
    return bool(list(TORCH_HUB_DIR.glob("snakers4_silero-vad*")))


def ensure_pretrained_models() -> None:
    if ECAPA_WEIGHTS.is_file() and _silero_ready():
        return

    if str(ECAPA_DIR) not in sys.path:
        sys.path.insert(0, str(ECAPA_DIR))

    from scripts.download_models import download_ecapa, download_silero

    if not ECAPA_WEIGHTS.is_file():
        download_ecapa()
    if not _silero_ready():
        download_silero()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/lang_id.py
"""Language identification — filter non-English / Vietnamese speech segments."""

from __future__ import annotations

import re
import threading
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

PathLike = str | Path

_VI_CHARS = re.compile(
    r"[\u0103\u0102\u00e2\u00c2\u00ea\u00ca\u00f4\u00d4\u01a1\u01a0\u01b0\u01af"
    r"\u00e0\u00e1\u00e3\u00e4\u00e5\u00e8\u00e9\u00eb\u00ec\u00ed\u00f2\u00f3"
    r"\u00f5\u00f9\u00fa\u00fd\u0111\u0110\u1ea0-\u1ef9]",
    re.UNICODE,
)

# Common Vietnamese words mis-heard by English-only Whisper
_VI_WORDS = re.compile(
    r"\b(V[AĂÂÁẠẢÃ]NG|KH[OÔÓỌỎÕ]NG|Đ[UƯÚỤỦŨ]|T[ÔƠÓỌỎÕ]I|C[OÔÓỌỎÕ]|M[UƯÚỤỦŨ]ON|"
    r"EM|ANH|CH[ÀAÁẠẢÃ]|KH[OÔ]|B[AĂÂÁẠẢÃ]N|NH[EÊÉẸẺẼ]|R[AĂÂÁẠẢÃ]T|N[ÀAÁẠẢÃ]Y|"
    r"G[IÍỊỈĨ]|H[OÔÓỌỎÕ]C|TI[ẾEÉẸẺẼ]NG|VI[EÊÉẸẺẼ]T)\b",
    re.IGNORECASE,
)

_classifier = None
_classifier_error: Optional[str] = None
_lock = threading.Lock()

# ISO 639-1 / SpeechBrain label fragments treated as Vietnamese
_VI_CODES = frozenset({"vi", "vie", "vietnamese"})


def _load_asr_lang_cfg() -> Dict[str, Any]:
    try:
        import yaml
        from paths import PRONUNCIATION_CONFIG

        with open(PRONUNCIATION_CONFIG, encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}
        return (cfg.get("asr") or {}).get("lang_id") or {}
    except Exception:
        return {}


def _normalize_lang(label: str) -> str:
    s = (label or "").strip().lower()
    if ":" in s:
        s = s.split(":", 1)[0].strip()
    if " " in s:
        s = s.split()[0]
    return s


def _get_classifier(device: str = "cpu"):
    global _classifier, _classifier_error
    if _classifier is not None:
        return _classifier
    if _classifier_error:
        return None
    with _lock:
        if _classifier is not None:
            return _classifier
        if _classifier_error:
            return None
        try:
            from speechbrain.inference.classifiers import EncoderClassifier

            from paths import ROOT

            savedir = ROOT / "pretrained_models" / "lang-id-voxlingua107-ecapa"
            _classifier = EncoderClassifier.from_hparams(
                source="speechbrain/lang-id-voxlingua107-ecapa",
                savedir=str(savedir),
                run_opts={"device": device},
            )
        except Exception as exc:
            _classifier_error = str(exc)
            print(f"[lang_id] Không tải được model LID: {exc}", flush=True)
            return None
        return _classifier


def detect_audio_language(
    audio_path: PathLike,
    *,
    device: str = "cpu",
) -> Tuple[Optional[str], float]:
    """Return (language_code, confidence) e.g. ('en', 0.92) or (None, 0)."""
    clf = _get_classifier(device)
    if clf is None:
        return None, 0.0
    try:
        out = clf.classify_file(str(audio_path))
        if isinstance(out, (list, tuple)):
            if len(out) >= 4:
                score = float(out[1]) if out[1] is not None else 0.0
                label = str(out[3])
            elif len(out) >= 2:
                score = float(out[0].max()) if hasattr(out[0], "max") else 0.0
                label = str(out[1][0]) if hasattr(out[1], "__getitem__") else str(out[1])
            else:
                return None, 0.0
        else:
            return None, 0.0
        return _normalize_lang(label), score
    except Exception as exc:
        print(f"[lang_id] classify_file lỗi: {exc}", flush=True)
        return None, 0.0


def text_looks_vietnamese(text: str) -> bool:
    if not text or not text.strip():
        return False
    if _VI_CHARS.search(text):
        return True
    if _VI_WORDS.search(text.upper()):
        return True
    return False


def is_vietnamese_segment(
    audio_path: PathLike,
    transcript: str = "",
    *,
    device: str = "cpu",
    cfg: Optional[Dict[str, Any]] = None,
) -> Tuple[bool, str]:
    """
    True if segment should be dropped as Vietnamese.
    Returns (drop, reason).
    """
    cfg = cfg or _load_asr_lang_cfg()
    if not cfg.get("enabled", True):
        return False, ""

    drop_langs = {_normalize_lang(x) for x in cfg.get("drop_languages", ["vi", "vie", "vietnamese"])}
    min_conf = float(cfg.get("min_confidence", 0.45))

    if cfg.get("text_vi_regex", True) and text_looks_vietnamese(transcript):
        return True, "text"

    lang, conf = detect_audio_language(audio_path, device=device)
    if lang is None:
        return False, ""

    if lang in drop_langs and conf >= min_conf:
        return True, f"audio:{lang}:{conf:.2f}"

    # SpeechBrain sometimes returns full name
    if any(v in lang for v in ("viet", "vietnam")) and conf >= min_conf:
        return True, f"audio:{lang}:{conf:.2f}"

    return False, ""


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/load_progress.py
"""Thread-safe model loading progress for the web UI."""

from __future__ import annotations

import threading
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

StepDef = Tuple[str, str, int]

STEPS: List[StepDef] = [
    ("config", "Đọc cấu hình", 3),
    ("pronunciation", "Chấm điểm: WavLM + Transformer", 55),
    ("pronunciation_ckpt", "Checkpoint pronunciation.pt", 18),
    ("whisper_proc", "Whisper: tokenizer + processor", 8),
    ("whisper_model", "Whisper: tải weights (small.en)", 16),
]


@dataclass
class LoadStep:
    id: str
    label: str
    status: str = "pending"
    detail: str = ""


class LoadProgress:
    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._steps: Dict[str, LoadStep] = {
            sid: LoadStep(sid, label) for sid, label, _ in STEPS
        }
        self._weights = {sid: w for sid, _, w in STEPS}
        self._current: Optional[str] = None
        self._error: Optional[str] = None
        self._done = False
        self._running_since: Optional[float] = None

    def reset(self) -> None:
        with self._lock:
            for sid, label, _ in STEPS:
                self._steps[sid] = LoadStep(sid, label)
            self._current = None
            self._error = None
            self._done = False
            self._running_since = None

    def start(self, step_id: str, detail: str = "") -> None:
        with self._lock:
            step = self._steps[step_id]
            step.status = "running"
            step.detail = detail
            self._current = step_id
            self._running_since = time.monotonic()
            label = step.label
        msg = f"{label}" + (f" — {detail}" if detail else "")
        print(f"[load] ▶ {msg}", flush=True)

    def finish(self, step_id: str, detail: str = "") -> None:
        with self._lock:
            step = self._steps[step_id]
            step.status = "done"
            if detail:
                step.detail = detail
            if self._current == step_id:
                self._current = None
                self._running_since = None
            label = step.label
        print(f"[load] ✓ {label}", flush=True)

    def fail_running(self, error: str) -> None:
        with self._lock:
            step_id = self._current
        if step_id:
            self.fail(step_id, error)

    def fail(self, step_id: str, error: str) -> None:
        with self._lock:
            self._steps[step_id].status = "error"
            self._steps[step_id].detail = error
            self._error = error
            label = self._steps[step_id].label
        print(f"[load] ✕ {label}: {error}", flush=True)

    def complete(self) -> None:
        with self._lock:
            self._done = True
            self._current = None
        print("[load] ✓ Tất cả model sẵn sàng", flush=True)

    def to_dict(self) -> dict:
        with self._lock:
            total = sum(self._weights.values())
            earned = 0.0
            steps_out = []
            all_done = True
            for sid, label, weight in STEPS:
                s = self._steps[sid]
                steps_out.append({
                    "id": sid,
                    "label": label,
                    "status": s.status,
                    "detail": s.detail,
                })
                if s.status != "done":
                    all_done = False
                if s.status == "done":
                    earned += weight
                elif s.status == "running":
                    earned += weight * 0.35

            if self._done or all_done:
                percent = 100
            else:
                percent = int(min(98, round(earned / total * 100)))

            current = self._steps[self._current] if self._current else None
            elapsed_sec = None
            if self._running_since is not None:
                elapsed_sec = int(time.monotonic() - self._running_since)

            return {
                "percent": percent,
                "current": self._current,
                "current_label": current.label if current else None,
                "current_detail": current.detail if current else None,
                "elapsed_sec": elapsed_sec,
                "steps": steps_out,
                "done": self._done,
                "error": self._error,
            }


progress = LoadProgress()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/pipeline.py
"""Unified inference: diarize 2 speakers → split sentences → score each segment."""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import yaml

from data.audio_preprocess import PreprocessConfig
from data.silence_split import export_diarization_clips, split_audio_file
from infer.lang_id import is_vietnamese_segment

from infer.pronunciation import Predictor
from infer.transcribe import transcribe_audio
from infer.device_utils import resolve_device
from paths import SPEAKER_DIARIZE_DIR, PRONUNCIATION_CONFIG, ROOT

SCORE_KEYS = ("accuracy", "fluency", "prosodic", "total")





def _build_dialogue(
    teacher_sentences: List[Dict[str, Any]],
    student_sentences: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """Chronological turns with teacher prompt attached before each student answer."""
    turns: List[Dict[str, Any]] = []
    for s in teacher_sentences:
        turns.append({
            "role": "teacher",
            "scored": "scores" in s,
            "start_sec": s.get("start_sec"),
            "end_sec": s.get("end_sec"),
            "transcript": s.get("transcript", ""),
            "audio": s.get("audio"),
            "scores": s.get("scores"),
            "errors": s.get("errors"),
        })
    for s in student_sentences:
        turns.append({
            "role": "student",
            "scored": True,
            "start_sec": s.get("start_sec"),
            "end_sec": s.get("end_sec"),
            "transcript": s.get("transcript", ""),
            "audio": s.get("audio"),
            "scores": s.get("scores"),
            "errors": s.get("errors"),
        })
    turns.sort(key=lambda t: (t.get("start_sec") or 0, 0 if t["role"] == "teacher" else 1))

    student_turns: List[Dict[str, Any]] = []
    last_teacher: Optional[Dict[str, Any]] = None
    for t in turns:
        if t["role"] == "teacher":
            last_teacher = t
        else:
            student_turns.append({
                **t,
                "teacher_prompt": last_teacher["transcript"] if last_teacher else None,
                "teacher_prompt_start_sec": last_teacher.get("start_sec") if last_teacher else None,
                "teacher_prompt_end_sec": last_teacher.get("end_sec") if last_teacher else None,
                "teacher_prompt_audio": last_teacher.get("audio") if last_teacher else None,
            })

    return {"turns": turns, "student_turns": student_turns}


def _avg_scores(items: List[Dict[str, Any]]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    for key in SCORE_KEYS:
        vals = [x["scores"][key] for x in items if x.get("scores", {}).get(key) is not None]
        if vals:
            out[key] = round(sum(vals) / len(vals), 2)
    return out


def _build_summary(
    sentences: List[Dict[str, Any]],
    *,
    pipeline: "SpeakingPipeline",
    feedback: bool,
    lang: Optional[str],
    speaker: Optional[str] = None,
    filtered_vi: int = 0,
    exchanges: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:
    summary_scores = _avg_scores(sentences)
    # Mỗi lượt nói (turn) = một dòng — không ghép liền
    lines = [s["transcript"].strip() for s in sentences if s.get("transcript", "").strip()]
    full_transcript = "\n".join(lines)
    summary_feedback = None
    summary_source = None
    pronunciation_feedback = None
    pronunciation_source = None

    if sentences:
        pronunciation_source = "local"

    if feedback and sentences:
        speaker_label = {
            "A": "Speaker A",
            "B": "Speaker B",
            "Student": "Student",
            "Teacher": "Teacher",
        }.get(speaker or "", None)
        if speaker_label is None and speaker:
            speaker_label = f"Speaker {speaker}"
        summary_feedback = ""
        summary_source = ""

    return {
        "sentence_count": len(sentences),
        "transcript": full_transcript,
        "transcript_lines": lines,
        "transcript_source": "whisper",
        "filtered_vi_count": filtered_vi,
        "scores": summary_scores,
        "pronunciation_feedback": pronunciation_feedback,
        "pronunciation_feedback_source": pronunciation_source,
        "feedback": summary_feedback,
        "feedback_source": summary_source,
    }


class SpeakingPipeline:
    """Diarize 2 speakers → silence-split → per-sentence scoring."""

    def __init__(
        self,
        config_path: str | Path | None = None,
        device: Optional[str] = None,
        pronunciation_ckpt: Optional[str] = None,
        enable_feedback: bool = True,
        load_progress: Optional[Any] = None,
    ):
        config_path = Path(config_path or PRONUNCIATION_CONFIG)
        if load_progress is not None:
            load_progress.start("config")
        with open(config_path, encoding="utf-8") as f:
            self.config = yaml.safe_load(f)
        if load_progress is not None:
            load_progress.finish("config")

        self.device = resolve_device(self.config, device)
        asr_cfg = self.config.setdefault("asr", {})
        if not asr_cfg.get("device"):
            asr_cfg["device"] = self.device
        self.preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
        wavlm_name = self.config.get("wavlm", {}).get("model_name", "microsoft/wavlm-large")
        self.pronunciation = Predictor(
            config_path,
            pronunciation_ckpt,
            self.device,
            load_progress=load_progress,
            model_step="pronunciation",
            ckpt_step="pronunciation_ckpt",
            wavlm_name=wavlm_name,
        )
        self.pronunciation.preprocess = self.preprocess
        
        if not SPEAKER_DIARIZE_DIR.is_dir():
            raise FileNotFoundError(f"speaker-diarize not found at {SPEAKER_DIARIZE_DIR}")
        if str(SPEAKER_DIARIZE_DIR) not in sys.path:
            sys.path.insert(0, str(SPEAKER_DIARIZE_DIR))
        from speaker_diarize.pipeline import TwoSpeakerSplitter
        
        self.diarizer = TwoSpeakerSplitter(device=self.device)
        from infer.transcribe import get_transcriber
        get_transcriber(load_progress)
        
        self.enable_feedback = enable_feedback
        self._lang_id_cfg = (self.config.get("asr") or {}).get("lang_id") or {}

    def _diarize_two_speakers(
        self,
        audio_path: Union[str, Path],
        output_dir: Optional[Union[str, Path]] = None,
        teacher_reference_path: Optional[Union[str, Path]] = None,
        teacher_embedding: Optional[Any] = None,
        student_embedding: Optional[Any] = None,
    ) -> Dict[str, Any]:
        result = self.diarizer.split_file(
            audio_path,
            output_dir,
            teacher_reference_path=teacher_reference_path,
            teacher_embedding=teacher_embedding,
            student_embedding=student_embedding,
        )
        out: Dict[str, Any] = {
            "segments": result.segments,
            "duration_sec": result.duration_sec,
            "teacher_cluster": result.teacher_cluster,
            "teacher_segments": result.teacher_segments,
            "student_segments": result.student_segments,
        }
        if result.teacher_path and result.student_path:
            out["teacher"] = result.teacher_path
            out["student"] = result.student_path
        return out

    def _collect_speaker_segments(
        self,
        output_dir: Path,
        speaker: str,
        *,
        source_audio: Union[str, Path],
        diarize_segments: list,
    ) -> List[Dict[str, Any]]:
        split_cfg = self.config.get("sentence_split") or {}
        track_preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
        track_preprocess.denoise = False
        merge_gap = float(split_cfg.get("diarization_merge_gap_sec", 0.2))

        segments = export_diarization_clips(
            source_audio,
            diarize_segments,
            output_dir,
            speaker,
            track_preprocess,
            merge_gap_sec=merge_gap,
            min_duration_sec=float(split_cfg.get("min_segment_sec", 0.2)),
            prefix=f"{speaker.lower()}_turn",
        )
        return segments

    def _process_segment(
        self,
        seg: Dict[str, Any],
        *,
        use_asr: bool,
        lang: Optional[str],
        score: bool = True,
        role: str = "student",
    ) -> Tuple[Optional[Dict[str, Any]], bool]:
        """Returns (sentence_dict, was_vi_filtered)."""
        if not use_asr:
            return None, False
        try:
            transcript = transcribe_audio(seg["path"])
        except ValueError:
            return None, False
        if not transcript.strip():
            return None, False

        drop_vi = False
        if score:
            drop_vi, reason = is_vietnamese_segment(
                seg["path"],
                transcript,
                device=self.device,
                cfg=self._lang_id_cfg,
            )
            if drop_vi:
                print(f"[lang_id] Bỏ đoạn tiếng Việt ({reason}): {transcript[:60]}…", flush=True)
                return None, True

        if not score:
            return {
                "index": seg["index"],
                "start_sec": seg["start_sec"],
                "end_sec": seg["end_sec"],
                "duration_sec": seg["duration_sec"],
                "audio": seg["path"],
                "turn_index": seg["index"],
                "transcript": transcript,
                "role": role,
                "scored": False,
            }, False

        try:
            track = self.assess_track(
                seg["path"], transcript, feedback=False, lang=lang,
                feedback_mode="local", truncate=False, apply_preprocess=False,
            )
        except ValueError:
            return None, False

        return {
            "index": seg["index"],
            "start_sec": seg["start_sec"],
            "end_sec": seg["end_sec"],
            "duration_sec": seg["duration_sec"],
            "audio": seg["path"],
            "turn_index": seg["index"],
            "role": role,
            "scored": True,
            **track,
        }, False

    def assess_track(
        self,
        audio: Union[str, Path],
        transcript: str,
        *,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        feedback_mode: str = "auto",
        truncate: bool = False,
        apply_preprocess: bool = True,
    ) -> Dict[str, Any]:
        fb = self.enable_feedback if feedback is None else feedback
        scores = self.pronunciation.predict(
            str(audio), transcript, fb, lang,
            feedback_mode=feedback_mode, truncate=truncate,
            apply_preprocess=apply_preprocess,
        )
        return {
            "audio": str(audio),
            "transcript": transcript,
            "scores": scores["scores"],
            "errors": scores["errors"],
            "alignments": scores.get("alignments"),
            "feedback": scores.get("feedback"),
            "feedback_source": scores.get("feedback_source"),
        }

    def _assess_speaker_sentences(
        self,
        *,
        output_dir: Path,
        speaker: str,
        source_audio: Union[str, Path],
        diarize_segments: list,
        use_asr: bool = True,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        score: bool = True,
        role: Optional[str] = None,
    ) -> Dict[str, Any]:
        fb = self.enable_feedback if feedback is None else feedback
        if not score:
            fb = False
        role = role or speaker
        segments = self._collect_speaker_segments(
            output_dir,
            speaker,
            source_audio=source_audio,
            diarize_segments=diarize_segments,
        )

        if not segments:
            raise ValueError(
                f"{role}: không tách được lượt nói nào từ audio "
                f"(kiểm tra diarization hoặc độ dài segment tối thiểu)"
            )

        sentences: List[Dict[str, Any]] = []
        filtered_vi = 0
        for seg in segments:
            item, was_vi = self._process_segment(
                seg, use_asr=use_asr, lang=lang, score=score, role=role,
            )
            if was_vi:
                filtered_vi += 1
            elif item:
                sentences.append(item)

        if not sentences and use_asr:
            if score:
                raise ValueError(
                    f"{role}: không còn đoạn tiếng Anh sau LID "
                    f"({len(segments)} turn, {filtered_vi} đoạn tiếng Việt đã loại, "
                    f"kiểm tra ASR hoặc transcript/CMUdict)"
                )
            raise ValueError(
                f"{role}: không transcribe được lượt nói nào "
                f"({len(segments)} turn, kiểm tra ASR)"
            )

        summary = _build_summary(
            sentences,
            pipeline=self,
            feedback=fb,
            lang=lang,
            speaker=speaker if score else None,
            filtered_vi=filtered_vi,
        )
        return {
            "role": role,
            "scored": score,
            "sentences": sentences,
            **summary,
        }

    def assess_conversation(
        self,
        audio: Union[str, Path],
        *,
        diarize_output_dir: Optional[Union[str, Path]] = None,
        use_asr: bool = True,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        teacher_voice: Optional[Union[str, Path]] = None,
        teacher_embedding: Optional[Any] = None,
        student_embedding: Optional[Any] = None,
        score_teacher: bool = False,
    ) -> Dict[str, Any]:
        """Diarize A/B → split each track by silence → score every sentence."""
        fb = self.enable_feedback if feedback is None else feedback
        audio = Path(audio)
        base_dir = Path(diarize_output_dir or audio.parent / f"{audio.stem}_split")
        split = self._diarize_two_speakers(
            audio,
            base_dir,
            teacher_reference_path=teacher_voice,
            teacher_embedding=teacher_embedding,
            student_embedding=student_embedding,
        )

        if split.get("teacher_segments") and split.get("student_segments"):
            teacher_dir = base_dir / "teacher_sentences"
            student_dir = base_dir / "student_sentences"
            teacher = self._assess_speaker_sentences(
                output_dir=teacher_dir,
                speaker="Teacher",
                source_audio=audio,
                diarize_segments=split.get("teacher_segments"),
                use_asr=use_asr,
                feedback=False,
                lang=lang,
                score=score_teacher,
                role="teacher",
            )
            student = self._assess_speaker_sentences(
                output_dir=student_dir,
                speaker="Student",
                source_audio=audio,
                diarize_segments=split.get("student_segments"),
                use_asr=use_asr,
                feedback=False,
                lang=lang,
                score=True,
                role="student",
            )
            dialogue = _build_dialogue(teacher["sentences"], student["sentences"])
            if fb and student.get("sentences"):
                lang_fb = _build_summary(
                    student["sentences"],
                    pipeline=self,
                    feedback=True,
                    lang=lang,
                    speaker="Student",
                    filtered_vi=student.get("filtered_vi_count", 0),
                    exchanges=dialogue["student_turns"],
                )
                student["feedback"] = lang_fb["feedback"]
                student["feedback_source"] = lang_fb["feedback_source"]
                student["pronunciation_feedback"] = lang_fb.get("pronunciation_feedback")
                student["pronunciation_feedback_source"] = lang_fb.get("pronunciation_feedback_source")
            return {
                "mode": "teacher_student",
                "source_audio": str(audio),
                "duration_sec": split["duration_sec"],
                "teacher": teacher,
                "student": student,
                "dialogue": dialogue,
            }

        speakers: Dict[str, Any] = {}
        for key, spk in [("A", "Speaker A"), ("B", "Speaker B")]:
            sent_dir = base_dir / f"speaker_{key}_sentences"
            speakers[key] = self._assess_speaker_sentences(
                output_dir=sent_dir,
                speaker=spk,
                use_asr=use_asr,
                feedback=fb,
                lang=lang,
                source_audio=audio,
                diarize_segments=split.get("segments"),
            )

        return {
            "mode": "diarize",
            "source_audio": str(audio),
            "duration_sec": split["duration_sec"],
            "speakers": speakers,
        }


def main():
    p = argparse.ArgumentParser(description="2-speaker speaking evaluation")
    p.add_argument("--audio", required=True)
    p.add_argument("--config", default=str(PRONUNCIATION_CONFIG))
    p.add_argument("--pronunciation-ckpt", default=None)
    p.add_argument("--no-feedback", action="store_true")
    p.add_argument("--lang", choices=["vi", "en"], default="vi")
    p.add_argument("--output", default=None)
    p.add_argument("--device", default=None)
    args = p.parse_args()

    pipe = SpeakingPipeline(
        args.config,
        args.device,
        args.pronunciation_ckpt,
        enable_feedback=not args.no_feedback,
    )
    result = pipe.assess_conversation(args.audio, lang=args.lang)

    text = json.dumps(result, ensure_ascii=False, indent=2, default=str)
    print(text)
    out = args.output or str(ROOT / "logs" / "result.json")
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    Path(out).write_text(text, encoding="utf-8")
    print(f"Saved: {out}")


if __name__ == "__main__":
    main()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/pronunciation.py
"""Pronunciation scoring inference (SpeechOcean762 model)."""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path
from typing import Any, Dict, Optional

import torch
import yaml
from dotenv import load_dotenv

from data.audio_preprocess import PreprocessConfig, load_waveform, truncate_waveform
from data.cmudict import CMUDict
from models.checkpoint_utils import load_model_weights
from models.pronunciation_model import PronunciationAssessmentModel
from models.pronunciation_scorer import PronunciationScorer
from paths import PRONUNCIATION_CONFIG
from dotenv import load_dotenv

load_dotenv()


class Predictor:
    def __init__(
        self,
        config_path: str | Path | None = None,
        checkpoint: Optional[str] = None,
        device: Optional[str] = None,
        load_progress: Optional[Any] = None,
        model_step: str = "pronunciation",
        ckpt_step: str = "pronunciation_ckpt",
        wavlm_name: str = "microsoft/wavlm-large",
    ):
        config_path = Path(config_path or PRONUNCIATION_CONFIG)
        with open(config_path, encoding="utf-8") as f:
            self.config = yaml.safe_load(f)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
        if load_progress is not None:
            load_progress.start(model_step, wavlm_name)
        try:
            self.model = PronunciationAssessmentModel(self.config).to(self.device)
            if load_progress is not None:
                load_progress.finish(model_step)
                load_progress.start(ckpt_step, "pronunciation.pt")
            try:
                self._ckpt_path = load_model_weights(
                    self.model, self.config, model_kind="pronunciation", explicit=checkpoint, device=self.device
                )
            except FileNotFoundError as e:
                print(f"Warning: {e}")
                self._ckpt_path = None
            except Exception as exc:
                if load_progress is not None:
                    load_progress.fail(ckpt_step, str(exc))
                raise
            if load_progress is not None:
                load_progress.finish(ckpt_step, str(self._ckpt_path or "không có checkpoint"))
        except Exception as exc:
            if load_progress is not None:
                load_progress.fail(model_step, str(exc))
            raise
        self.model.eval()
        self.sr = self.preprocess.sample_rate
        inf = self.config.get("inference") or {}
        md = inf.get("max_duration_sec")
        if md is None and "max_duration_sec" not in inf:
            ds = self.config.get("train", {}).get("dataset", {})
            md = ds.get("max_duration_sec")
        self.max_duration_sec = None if md is None or md <= 0 else float(md)
        self.cmudict = CMUDict(self.config["paths"].get("cmudict_path"))
        mt = self.config["multitask"]
        self.scorer = PronunciationScorer(mt.get("score_scale", 2.0), self.config.get("scorer", {}).get("weights"))

    def _phones_from_text(self, text: str):
        groups = self.cmudict.words_to_phoneme_groups(text)
        tokens, words, ranges = [], [], []
        for g in groups:
            s = len(tokens)
            tokens.extend(g["phones"])
            words.append(g["word"])
            ranges.append((s, len(tokens)))
        return tokens, words, ranges

    @torch.no_grad()
    def predict(
        self,
        audio: str,
        transcript: str,
        feedback: bool = True,
        lang: Optional[str] = None,
        feedback_mode: str = "auto",
        truncate: bool = False,
        *,
        apply_preprocess: bool = True,
    ) -> Dict[str, Any]:
        wav = load_waveform(audio, self.preprocess, apply_preprocess=apply_preprocess)
        truncated = False
        if truncate and self.max_duration_sec:
            wav, truncated = truncate_waveform(wav, self.sr, self.max_duration_sec)
        tokens, words, ranges = self._phones_from_text(transcript)
        if not tokens:
            raise ValueError(
                f"Không tra được phoneme cho transcript (CMUdict): {transcript!r}"
            )
        out = self.model(
            wav.unsqueeze(0).to(self.device),
            torch.tensor([wav.shape[0]], device=self.device),
            [tokens],
            [ranges],
            return_alignments=True,
        )
        pred = out["predictions"][0]
        scores = self.scorer.aggregate_utterance(pred)
        scores["final"] = self.scorer.final_score(pred)
        errors = self.scorer.find_errors(pred, tokens, words, ranges)
        result = {
            "transcript": transcript,
            "scores": scores,
            "errors": errors,
            "truncated": truncated,
            "max_duration_sec": self.max_duration_sec,
            "alignments": [
                {
                    "phoneme": a.phoneme,
                    "start_frame": a.start_frame,
                    "end_frame": a.end_frame,
                    "confidence": a.confidence,
                }
                for a in (out.get("alignments") or [[]])[0]
            ],
            "feedback": None,
            "feedback_source": None,
        }
        if feedback:
            # Feedback is handled in the Colab notebook directly
            pass
        return result


def main():
    p = argparse.ArgumentParser(description="Pronunciation scoring only")
    p.add_argument("--audio", required=True)
    p.add_argument("--transcript", required=True)
    p.add_argument("--config", default=str(PRONUNCIATION_CONFIG))
    p.add_argument("--checkpoint", default=None)
    p.add_argument("--no-feedback", action="store_true")
    p.add_argument("--output", default=None)
    args = p.parse_args()
    r = Predictor(args.config, args.checkpoint).predict(args.audio, args.transcript, not args.no_feedback)
    text = json.dumps(r, ensure_ascii=False, indent=2)
    print(text)
    if args.output:
        Path(args.output).write_text(text, encoding="utf-8")


if __name__ == "__main__":
    main()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/transcribe.py
"""Speech-to-text for automatic transcript (Whisper)."""

from __future__ import annotations

import re
import threading
from pathlib import Path
from typing import Any, Dict, Optional, Union

import torch
import torchaudio
from data.audio_preprocess import load_audio_file
import yaml

from infer.device_utils import resolve_device
from paths import PRONUNCIATION_CONFIG

PathLike = Union[str, Path]
WHISPER_SR = 16000
WHISPER_CHUNK_SEC = 30.0

_transcriber: Optional["WhisperTranscriber"] = None
_transcriber_error: Optional[str] = None
_lock = threading.Lock()


def _normalize_transcript(text: str) -> str:
    text = re.sub(r"[^\w\s']", " ", text.upper())
    return " ".join(text.split())


def _load_asr_config() -> Dict[str, Any]:
    if PRONUNCIATION_CONFIG.is_file():
        with open(PRONUNCIATION_CONFIG, encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}
        return cfg.get("asr") or {}
    return {}


def _parse_dtype(name: Optional[str], device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    if not name or name == "float16":
        return torch.float16
    if name == "bfloat16":
        return torch.bfloat16
    return torch.float32


class WhisperTranscriber:
    """Whisper ASR — default: whisper-small.en (English, CPU-friendly)."""

    def __init__(
        self,
        model_name: str = "openai/whisper-small.en",
        language: str = "en",
        device: Optional[str] = None,
        load_progress: Optional[Any] = None,
        torch_dtype: Optional[str] = "float16",
        max_new_tokens: int = 448,
    ):
        from transformers import WhisperForConditionalGeneration, WhisperProcessor

        asr_cfg = _load_asr_config()
        self.device = torch.device(
            device or resolve_device({"asr": asr_cfg}, asr_cfg.get("device"))
        )
        self.language = language
        self.max_new_tokens = max_new_tokens
        self.model_name = model_name
        dtype = _parse_dtype(torch_dtype, self.device)

        if load_progress is not None:
            load_progress.start("whisper_proc", model_name)
        self.processor = WhisperProcessor.from_pretrained(model_name)
        if load_progress is not None:
            load_progress.finish("whisper_proc", model_name)
            load_progress.start("whisper_model", model_name)

        load_kw: Dict[str, Any] = {}
        if self.device.type == "cuda" and dtype != torch.float32:
            load_kw["torch_dtype"] = dtype

        self.model = WhisperForConditionalGeneration.from_pretrained(model_name, **load_kw)
        self.model.to(self.device)
        self.model.eval()

        if load_progress is not None:
            load_progress.finish("whisper_model", model_name)

    def _generate(self, input_features: torch.Tensor) -> torch.Tensor:
        name = self.model_name.lower()
        max_tokens = self.max_new_tokens

        if name.endswith(".en"):
            # English-only checkpoints: no language/task prefix tokens
            return self.model.generate(input_features, max_new_tokens=max_tokens)

        if "large-v3" in name or "large-v2" in name:
            return self.model.generate(
                input_features,
                max_new_tokens=max_tokens,
                language=self.language,
                task="transcribe",
            )

        # Multilingual (medium/small/tiny): forced_decoder_ids uses ~4 prefix tokens
        max_tokens = min(max_tokens, 444)
        return self.model.generate(
            input_features,
            max_new_tokens=max_tokens,
            forced_decoder_ids=self.processor.get_decoder_prompt_ids(
                language=self.language, task="transcribe"
            ),
        )

    def _transcribe_waveform(self, wav: torch.Tensor) -> str:
        inputs = self.processor(wav.numpy(), sampling_rate=WHISPER_SR, return_tensors="pt")
        input_features = inputs.input_features.to(self.device)
        if self.device.type == "cuda" and input_features.dtype == torch.float32:
            input_features = input_features.half()
        ids = self._generate(input_features)
        return self.processor.batch_decode(ids, skip_special_tokens=True)[0]

    @torch.inference_mode()
    def transcribe(self, audio_path: PathLike) -> str:
        wav, sr = load_audio_file(audio_path)
        wav = wav.mean(0)
        if sr != WHISPER_SR:
            wav = torchaudio.functional.resample(wav, sr, WHISPER_SR)

        chunk_samples = int(WHISPER_CHUNK_SEC * WHISPER_SR)
        if wav.shape[0] <= chunk_samples:
            text = self._transcribe_waveform(wav)
        else:
            parts = []
            for start in range(0, wav.shape[0], chunk_samples):
                parts.append(self._transcribe_waveform(wav[start : start + chunk_samples]))
            text = " ".join(parts)

        normalized = _normalize_transcript(text)
        if not normalized:
            raise ValueError("Không nhận diện được lời nói trong audio")
        return normalized


def get_transcriber(load_progress: Optional[Any] = None) -> WhisperTranscriber:
    global _transcriber, _transcriber_error
    if _transcriber is not None:
        return _transcriber
    if _transcriber_error:
        raise RuntimeError(_transcriber_error)
    with _lock:
        if _transcriber is not None:
            return _transcriber
        if _transcriber_error:
            raise RuntimeError(_transcriber_error)
        try:
            asr = _load_asr_config()
            _transcriber = WhisperTranscriber(
                model_name=asr.get("model_name", "openai/whisper-small.en"),
                language=asr.get("language", "en"),
                device=asr.get("device"),
                load_progress=load_progress,
                torch_dtype=asr.get("torch_dtype", "float16"),
                max_new_tokens=int(asr.get("max_new_tokens", 448)),
            )
        except Exception as exc:
            _transcriber_error = str(exc)
            if load_progress is not None:
                load_progress.fail_running(str(exc))
            raise RuntimeError(_transcriber_error) from exc
        return _transcriber


def transcribe_audio(audio_path: PathLike) -> str:
    return get_transcriber().transcribe(audio_path)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/__init__.py
"""2-speaker audio splitting with ERes2Net-Large."""

from speaker_diarize.denoise import denoise_with_deepfilternet, level_audio_to_target
from speaker_diarize.embedding import ERes2NetEmbedder
from speaker_diarize.pipeline import DiarizationSegment, SplitResult, TwoSpeakerSplitter
from speaker_diarize.segmentation import RmsVad

__all__ = [
    "DiarizationSegment",
    "ERes2NetEmbedder",
    "RmsVad",
    "denoise_with_deepfilternet",
    "level_audio_to_target",
    "SplitResult",
    "TwoSpeakerSplitter",
]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/audio_io.py
"""Load and save audio files."""

from __future__ import annotations

from pathlib import Path

import numpy as np

SAMPLE_RATE = 16000


def load_audio(path: str | Path) -> tuple[np.ndarray, int]:
    """Load mono float32 audio, resampled to 16 kHz if needed."""
    import soundfile as sf
    import torchaudio

    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
    mono = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(
            __import__("torch").from_numpy(mono).unsqueeze(0), sr, SAMPLE_RATE
        )
        mono = wav.squeeze(0).numpy()
    return mono.astype(np.float32), SAMPLE_RATE


def save_audio(path: str | Path, audio: np.ndarray, sample_rate: int = SAMPLE_RATE) -> None:
    import soundfile as sf

    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), audio, sample_rate)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/clustering.py
"""Online 2-speaker clustering with EMA centroid updates."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

DEFAULT_THRESHOLD = 0.25  # SpeechBrain cosine-distance verification default


@dataclass
class ClusterAssignment:
    speaker: str
    confidence: float
    calibrating: bool


class TwoSpeakerClusterer:
    """Assign embeddings to Speaker A / B with online centroid updates."""

    def __init__(
        self,
        num_speakers: int = 2,
        threshold: float = DEFAULT_THRESHOLD,
        ema_alpha: float = 0.15,
        min_separation: float = 0.15,
    ) -> None:
        if num_speakers != 2:
            raise ValueError("Only 2-speaker mode is supported")
        self.threshold = threshold
        self.ema_alpha = ema_alpha
        self.min_separation = min_separation
        self._centroids: list[np.ndarray | None] = [None, None]
        self._labels = ["Speaker A", "Speaker B"]
        self._pending: list[np.ndarray] = []
        self._calibrated = False

    @property
    def calibrating(self) -> bool:
        return not self._calibrated

    def assign(self, embedding: np.ndarray) -> ClusterAssignment:
        emb = self._normalize(embedding)

        if not self._calibrated:
            return self._calibrate(emb)

        sim_a = float(np.dot(emb, self._centroids[0]))
        sim_b = float(np.dot(emb, self._centroids[1]))
        dist_a = 1.0 - sim_a
        dist_b = 1.0 - sim_b

        if dist_a <= dist_b:
            idx, confidence = 0, 1.0 - dist_a
        else:
            idx, confidence = 1, 1.0 - dist_b

        self._update_centroid(idx, emb)
        return ClusterAssignment(
            speaker=self._labels[idx],
            confidence=float(np.clip(confidence, 0.0, 1.0)),
            calibrating=False,
        )

    def _calibrate(self, emb: np.ndarray) -> ClusterAssignment:
        self._pending.append(emb)

        if self._centroids[0] is None:
            self._centroids[0] = emb.copy()
            return ClusterAssignment(speaker="calibrating", confidence=0.0, calibrating=True)

        dist_to_a = 1.0 - float(np.dot(emb, self._centroids[0]))
        if dist_to_a > self.threshold and self._centroids[1] is None:
            self._centroids[1] = emb.copy()
            if self._centroid_distance() >= self.min_separation:
                self._calibrated = True
            return ClusterAssignment(speaker="calibrating", confidence=0.0, calibrating=True)

        idx = 0 if dist_to_a <= self.threshold else 1
        if self._centroids[idx] is None:
            self._centroids[idx] = emb.copy()
        else:
            self._update_centroid(idx, emb)

        if self._centroids[0] is not None and self._centroids[1] is not None:
            if self._centroid_distance() >= self.min_separation:
                self._calibrated = True
                sim_a = float(np.dot(emb, self._centroids[0]))
                sim_b = float(np.dot(emb, self._centroids[1]))
                if sim_a >= sim_b:
                    return ClusterAssignment(speaker=self._labels[0], confidence=sim_a, calibrating=False)
                return ClusterAssignment(speaker=self._labels[1], confidence=sim_b, calibrating=False)

        return ClusterAssignment(speaker="calibrating", confidence=0.0, calibrating=True)

    def _centroid_distance(self) -> float:
        if self._centroids[0] is None or self._centroids[1] is None:
            return 0.0
        return 1.0 - float(np.dot(self._centroids[0], self._centroids[1]))

    def _update_centroid(self, idx: int, emb: np.ndarray) -> None:
        current = self._centroids[idx]
        if current is None:
            self._centroids[idx] = emb.copy()
            return
        updated = (1.0 - self.ema_alpha) * current + self.ema_alpha * emb
        self._centroids[idx] = self._normalize(updated)

    @staticmethod
    def _normalize(vec: np.ndarray) -> np.ndarray:
        norm = np.linalg.norm(vec)
        if norm == 0:
            return vec.astype(np.float32)
        return (vec / norm).astype(np.float32)


def batch_cluster_two(embeddings: list[np.ndarray]) -> list[int]:
    """Assign N embeddings to cluster 0 or 1 (Speaker A / B)."""
    if not embeddings:
        return []
    if len(embeddings) == 1:
        return [0]

    embs = np.stack([TwoSpeakerClusterer._normalize(e) for e in embeddings])
    # Seed: first embedding + furthest from it
    sims = embs @ embs[0]
    seed_b = int(np.argmin(sims))
    centers = np.stack([embs[0], embs[seed_b]])

    for _ in range(5):
        labels = []
        for e in embs:
            labels.append(0 if float(centers[0] @ e) >= float(centers[1] @ e) else 1)
        labels_arr = np.array(labels)
        for k in (0, 1):
            members = embs[labels_arr == k]
            if len(members) > 0:
                c = members.mean(axis=0)
                norm = np.linalg.norm(c)
                if norm > 0:
                    centers[k] = c / norm

    return labels


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/denoise.py
import numpy as np
import torch
import warnings

try:
    from df.enhance import enhance, init_df, load_audio as df_load_audio, save_audio
    _df_model, _df_state = None, None
except ImportError:
    pass

def _get_df_model():
    global _df_model, _df_state
    if _df_model is None or _df_state is None:
        _df_model, _df_state, _ = init_df()
    return _df_model, _df_state

def denoise_with_deepfilternet(audio: np.ndarray, sr: int) -> tuple[np.ndarray, int]:
    try:
        model, df_state = _get_df_model()
        if audio.ndim == 1:
            audio = audio[np.newaxis, :]
        audio_tensor = torch.from_numpy(audio).float()
        
        enhanced = enhance(model, df_state, audio_tensor)
        return enhanced.squeeze().numpy(), df_state.sr()
    except Exception as e:
        warnings.warn(f"DeepFilterNet failed, returning original audio: {e}")
        return audio, sr

def level_audio_to_target(audio: np.ndarray, sr: int, target_db: float = -20.0) -> np.ndarray:
    rms = np.sqrt(np.mean(audio**2))
    if rms == 0:
        return audio
    current_db = 20 * np.log10(rms + 1e-9)
    gain = 10 ** ((target_db - current_db) / 20)
    return audio * gain


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/embedding.py
import numpy as np
import torch
import warnings

# Tắt cảnh báo librosa/modelscope
warnings.filterwarnings("ignore")

SAMPLE_RATE = 16000

import os
DEFAULT_MODEL_ID = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "pretrained_models", "speech_eres2net_large_200k_sv_zh-cn_16k-common")

class ERes2NetEmbedder:
    """Wrap ModelScope ERes2NetV2 for speaker embeddings."""

    def __init__(self, device: str = "cuda", model_id: str = DEFAULT_MODEL_ID) -> None:
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model_id = model_id
        from modelscope.pipelines import pipeline
        
        # Initialize pipeline
        print(f"[{model_id.split('/')[-1]}] Đang tải mô hình từ ModelScope...")
        self.sv_pipeline = pipeline(
            task='speaker-verification',
            model=model_id,
            device=self.device
        )
        print(f"[{model_id.split('/')[-1]}] Tải mô hình thành công!")

    def embed(self, audio: np.ndarray | torch.Tensor, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
        if isinstance(audio, np.ndarray):
            wav = audio.astype(np.float32)
        else:
            wav = audio.cpu().numpy().astype(np.float32)

        # Đảm bảo là mảng 1D
        while wav.ndim > 1:
            wav = wav[0]
            
        if len(wav) < int(0.2 * SAMPLE_RATE):
            raise ValueError("Audio too short for embedding (need >= 0.2s)")

        # Thử cách đưa qua file tạm để tương thích tốt nhất với pipeline
        import soundfile as sf
        import tempfile
        import os
        
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            temp_path = f.name
        try:
            sf.write(temp_path, wav, sample_rate)
            res = self.sv_pipeline([temp_path], output_emb=True)
            if isinstance(res, dict) and 'embs' in res:
                vec = res['embs']
            elif isinstance(res, dict) and 'text' in res:
                # Fallback if the pipeline returns text dict
                vec = res.get('embs', res)
            else:
                vec = res
                
            if isinstance(vec, list):
                vec = np.array(vec)
            vec = np.squeeze(vec).astype(np.float32)
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec /= norm
            return vec
        finally:
            if os.path.exists(temp_path):
                try:
                    os.remove(temp_path)
                except:
                    pass


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/pipeline.py
"""Diarization pipeline using ERes2Net-Large."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np

from speaker_diarize.audio_io import load_audio, save_audio
from speaker_diarize.clustering import TwoSpeakerClusterer, batch_cluster_two
from speaker_diarize.embedding import ERes2NetEmbedder
from speaker_diarize.segmentation import RmsVad, SlidingWindowBuffer, SpeechWindow

LABELS = ("Speaker A", "Speaker B")
ROLE_TEACHER = "Teacher"
ROLE_STUDENT = "Student"


@dataclass
class DiarizationSegment:
    start: float
    end: float
    speaker: str
    confidence: float
    teacher_score: float | None = None
    student_score: float | None = None


@dataclass
class SplitResult:
    segments: list[DiarizationSegment]
    duration_sec: float
    teacher_cluster: int | None = None
    teacher_segments: list[DiarizationSegment] | None = None
    student_segments: list[DiarizationSegment] | None = None
    teacher_path: Path | None = None
    student_path: Path | None = None


class TwoSpeakerSplitter:
    """Diarize a 2-speaker recording and export separate audio tracks."""

    def __init__(
        self,
        device: str = "cpu",
        embedder: ERes2NetEmbedder | None = None,
        vad: RmsVad | None = None,
        cluster_window_sec: float = 1.5,
        boundary_window_sec: float = 0.5,
        min_speech_sec: float = 0.25,
        min_segment_sec: float = 0.3,
        merge_gap_sec: float = 0.5,
        step_sec: float = 0.5,
        boundary_step_sec: float = 0.1,
    ) -> None:
        self.embedder = embedder or ERes2NetEmbedder(device=device)
        self.vad = vad or RmsVad()
        self.min_segment_sec = min_segment_sec
        self.merge_gap_sec = merge_gap_sec
        self.buffer = SlidingWindowBuffer(window_sec=cluster_window_sec, step_sec=step_sec)
        self.boundary_buffer = SlidingWindowBuffer(
            window_sec=boundary_window_sec, 
            step_sec=boundary_step_sec, 
            min_speech_sec=min_speech_sec
        )

    def split_file(
        self,
        input_path: str | Path,
        output_dir: str | Path | None = None,
        *,
        teacher_reference_path: str | Path | None = None,
        teacher_embedding: np.ndarray | None = None,
        student_embedding: np.ndarray | None = None,
        apply_denoise: bool = True,
    ) -> SplitResult:
        input_path = Path(input_path)

        teacher_emb = teacher_embedding
        if teacher_emb is None and teacher_reference_path:
            teacher_emb = self._embed_reference(teacher_reference_path)
        elif teacher_emb is not None:
            pass

        audio, sr = load_audio(input_path)
        


        segments, teacher_cluster = self._diarize(
            audio, sr, teacher_emb=teacher_emb, student_emb=student_embedding
        )

        teacher_segments = [s for s in segments if s.speaker == ROLE_TEACHER] if teacher_cluster is not None else None
        student_segments = [s for s in segments if s.speaker == ROLE_STUDENT] if teacher_cluster is not None else None

        teacher_path = student_path = None
        if teacher_segments is not None and student_segments is not None:
            teacher_track = np.zeros_like(audio)
            student_track = np.zeros_like(audio)
            for s in teacher_segments:
                teacher_track[int(s.start * sr):int(s.end * sr)] = audio[int(s.start * sr):int(s.end * sr)]
            for s in student_segments:
                student_track[int(s.start * sr):int(s.end * sr)] = audio[int(s.start * sr):int(s.end * sr)]
            
            if output_dir is not None:
                out_dir = Path(output_dir)
            else:
                out_dir = input_path.parent / f"{input_path.stem}_split"
            out_dir.mkdir(parents=True, exist_ok=True)
            teacher_path = out_dir / f"{input_path.stem}_teacher.wav"
            student_path = out_dir / f"{input_path.stem}_student.wav"
            csv_path = out_dir / f"{input_path.stem}_cosine_scores.csv"
            
            save_audio(teacher_path, teacher_track, sr)
            save_audio(student_path, student_track, sr)
            
            # Lưu file CSV hiển thị điểm cosine
            with open(csv_path, "w", encoding="utf-8") as f:
                f.write("Start,End,Assigned_Speaker,Teacher_Score,Student_Score\n")
                for s in segments:
                    ts = f"{s.teacher_score:.4f}" if s.teacher_score is not None else "N/A"
                    ss = f"{s.student_score:.4f}" if s.student_score is not None else "N/A"
                    f.write(f"{s.start:.2f},{s.end:.2f},{s.speaker},{ts},{ss}\n")

        return SplitResult(
            segments=segments,
            duration_sec=len(audio) / sr,
            teacher_cluster=teacher_cluster,
            teacher_segments=teacher_segments,
            student_segments=student_segments,
            teacher_path=teacher_path,
            student_path=student_path,
        )

    def _embed_reference(self, reference_path: str | Path, apply_denoise: bool = True) -> np.ndarray:
        audio, sr = load_audio(reference_path)
        

        
        embs: list[np.ndarray] = []
        for window in self.buffer.iter_windows(audio):
            if not self.vad.is_speech(window.audio):
                continue
            try:
                embs.append(self.embedder.embed(window.audio))
            except ValueError:
                continue
        if embs:
            vec = np.mean(embs, axis=0).astype(np.float32)
        else:
            vec = self.embedder.embed(audio)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec /= norm
        return vec

    def _diarize(
        self,
        audio: np.ndarray,
        sample_rate: int,
        teacher_emb: np.ndarray | None = None,
        student_emb: np.ndarray | None = None,
    ) -> tuple[list[DiarizationSegment], int | None]:
        windows: list[SpeechWindow] = []
        embeddings: list[np.ndarray] = []

        for window in self.buffer.iter_windows(audio):
            if not self.vad.is_speech(window.audio):
                continue
            try:
                embeddings.append(self.embedder.embed(window.audio))
                windows.append(window)
            except ValueError:
                continue

        if not windows:
            raise ValueError("Không phát hiện giọng nói trong file audio.")

        labels = batch_cluster_two(embeddings)
        centers = self._cluster_centers(embeddings, labels)
        n_clusters = len(set(labels))

        teacher_cluster: int | None = None
        if n_clusters == 1:
            teacher_cluster = 0
        else:
            if teacher_emb is not None and len(teacher_emb) != len(centers[0]):
                import warnings
                warnings.warn(f"Teacher embedding dimension mismatch: expected {len(centers[0])}, got {len(teacher_emb)}. Ignoring teacher embedding.")
                teacher_emb = None
            if student_emb is not None and len(student_emb) != len(centers[0]):
                import warnings
                warnings.warn(f"Student embedding dimension mismatch: expected {len(centers[0])}, got {len(student_emb)}. Ignoring student embedding.")
                student_emb = None

            sim_t0 = float(np.dot(centers[0], teacher_emb)) if teacher_emb is not None else 0.0
            sim_t1 = float(np.dot(centers[1], teacher_emb)) if teacher_emb is not None else 0.0
            sim_s0 = float(np.dot(centers[0], student_emb)) if student_emb is not None else 0.0
            sim_s1 = float(np.dot(centers[1], student_emb)) if student_emb is not None else 0.0

            score_a = sim_t0 + sim_s1
            score_b = sim_t1 + sim_s0

            teacher_cluster = 0 if score_a >= score_b else 1
            
            print(f"\n[Mapping Similarity]")
            print(f"- Giáo viên (Teacher) so với Cluster 0: {sim_t0:.3f}")
            print(f"- Giáo viên (Teacher) so với Cluster 1: {sim_t1:.3f}")
            print(f"- Học sinh (Student)  so với Cluster 0: {sim_s0:.3f}")
            print(f"- Học sinh (Student)  so với Cluster 1: {sim_s1:.3f}")
            print(f"=> Quyết định: Gán Giáo viên = Cluster {teacher_cluster}, Học sinh = Cluster {1 - teacher_cluster}")


        n = len(audio)
        votes_a = np.zeros(n, dtype=np.float32)
        votes_b = np.zeros(n, dtype=np.float32)

        for window in self.boundary_buffer.iter_windows(audio):
            if not self.vad.is_speech(window.audio, min_ratio=0.1):
                continue
            try:
                emb = self.embedder.embed(window.audio)
                conf_a = float(np.clip(np.dot(emb, centers[0]), 0.0, 1.0))
                conf_b = float(np.clip(np.dot(emb, centers[1]), 0.0, 1.0))
                
                label = 0 if conf_a >= conf_b else 1
                confidence = conf_a if label == 0 else conf_b
                
                s, e = window.start_sample, window.end_sample
                if label == 0:
                    votes_a[s:e] += confidence
                else:
                    votes_b[s:e] += confidence
            except ValueError:
                continue

        stamps = self.vad.get_timestamps(audio, sample_rate=sample_rate)
        is_speech = np.zeros(n, dtype=bool)
        pad_samples = int(0.2 * sample_rate)  # Expand speech by 200ms to avoid missing soft ends
        for stamp in stamps:
            s_idx = max(0, stamp["start"] - pad_samples)
            e_idx = min(n, stamp["end"] + pad_samples)
            is_speech[s_idx:e_idx] = True
            
        pred_a = (votes_a >= votes_b) & (votes_a > 0) & is_speech
        pred_b = (votes_b > votes_a) & is_speech
        
        def get_blocks(mask):
            mask_int = np.concatenate(([0], mask.astype(int), [0]))
            diff = np.diff(mask_int)
            starts = np.where(diff == 1)[0]
            ends = np.where(diff == -1)[0]
            return starts, ends
            
        starts_a, ends_a = get_blocks(pred_a)
        starts_b, ends_b = get_blocks(pred_b)
        
        raw_segments = []
        for s, e in zip(starts_a, ends_a):
            raw_segments.append({
                "start": float(s) / sample_rate,
                "end": float(e) / sample_rate,
                "cluster": 0
            })
        for s, e in zip(starts_b, ends_b):
            raw_segments.append({
                "start": float(s) / sample_rate,
                "end": float(e) / sample_rate,
                "cluster": 1
            })
            
        raw_segments.sort(key=lambda x: x["start"])
        
        # 1. Filter out excessively short segments to prevent flickering
        raw_segments = [s for s in raw_segments if (s["end"] - s["start"]) >= self.min_segment_sec]
        
        # 2. Merge same-speaker segments that are close to each other
        merged_segments = []
        for seg in raw_segments:
            if not merged_segments:
                merged_segments.append(seg)
                continue
                
            last = merged_segments[-1]
            if last["cluster"] == seg["cluster"] and (seg["start"] - last["end"] < self.merge_gap_sec):
                last["end"] = seg["end"]
            else:
                merged_segments.append(seg)
                
        final_segments = []
        for s in merged_segments:
            t_score = None
            s_score = None
            
            if teacher_emb is not None and student_emb is not None:
                # 2nd-pass Refinement: Chấm điểm trực tiếp từng phân đoạn để gán nhãn
                chunk = audio[int(s["start"]*sample_rate) : int(s["end"]*sample_rate)]
                
                # Thay vì nhúng cả 1 đoạn dài (có thể lẫn khoảng lặng/thở làm loãng vector), 
                # ta dùng chung buffer (1.5s) và VAD lọc tiếng nói hệt như cách lấy mẫu Reference để vector chuẩn nhất.
                chunk_embs = []
                for window in self.buffer.iter_windows(chunk):
                    if self.vad.is_speech(window.audio):
                        try:
                            chunk_embs.append(self.embedder.embed(window.audio))
                        except ValueError:
                            pass
                
                try:
                    if chunk_embs:
                        seg_emb = np.mean(chunk_embs, axis=0)
                    else:
                        seg_emb = self.embedder.embed(chunk)
                        
                    norm = np.linalg.norm(seg_emb)
                    if norm > 0: seg_emb /= norm
                    
                    t_score = float(np.dot(seg_emb, teacher_emb))
                    s_score = float(np.dot(seg_emb, student_emb))
                    
                    # Trust K-Means clustering which naturally separates the two speakers in this audio
                    role = ROLE_TEACHER if s["cluster"] == teacher_cluster else ROLE_STUDENT
                    
                    # Filter out segments with poor cosine similarity (< 0.4)
                    if role == ROLE_TEACHER and t_score < 0.4:
                        continue
                    if role == ROLE_STUDENT and s_score < 0.4:
                        continue
                        
                except ValueError:
                    # Fallback for too short segments
                    role = ROLE_TEACHER if s["cluster"] == teacher_cluster else ROLE_STUDENT
            else:
                if teacher_cluster is not None:
                    role = ROLE_TEACHER if s["cluster"] == teacher_cluster else ROLE_STUDENT
                else:
                    role = LABELS[s["cluster"]]
                
            final_segments.append(DiarizationSegment(
                start=s["start"], end=s["end"], speaker=role, confidence=1.0,
                teacher_score=t_score, student_score=s_score
            ))

        return final_segments, teacher_cluster

    @staticmethod
    def _cluster_centers(embeddings: list[np.ndarray], labels: list[int]) -> list[np.ndarray]:
        centers = []
        for k in (0, 1):
            members = [embeddings[i] for i, lb in enumerate(labels) if lb == k]
            if not members:
                centers.append(TwoSpeakerClusterer._normalize(embeddings[0]))
            else:
                c = np.mean(members, axis=0)
                centers.append(TwoSpeakerClusterer._normalize(c))
        return centers


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/segmentation.py
"""Voice activity detection and speech window extraction."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

NOISE_FLOOR_DB = -65.0

SAMPLE_RATE = 16000


@dataclass
class SpeechWindow:
    audio: np.ndarray
    start_sample: int
    end_sample: int

    @property
    def start_sec(self) -> float:
        return self.start_sample / SAMPLE_RATE

    @property
    def end_sec(self) -> float:
        return self.end_sample / SAMPLE_RATE


class RmsVad:
    """VAD based on RMS noise floor."""

    def __init__(self, threshold_db: float = NOISE_FLOOR_DB) -> None:
        self.threshold_lin = 10.0 ** (threshold_db / 20.0)

    def is_speech(self, audio: np.ndarray, sample_rate: int = SAMPLE_RATE, min_ratio: float = 0.3) -> bool:
        if len(audio) == 0:
            return False
        frame_len = int(sample_rate * 0.05)
        if len(audio) < frame_len:
            rms = np.sqrt(np.mean(audio ** 2) + 1e-12)
            return bool(rms > self.threshold_lin)
            
        n_frames = len(audio) // frame_len
        trimmed = audio[: n_frames * frame_len]
        frames = trimmed.reshape(n_frames, frame_len)
        rms = np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12)
        speech_frames = np.sum(rms > self.threshold_lin)
        return (speech_frames / n_frames) >= min_ratio

    def get_timestamps(self, audio: np.ndarray, sample_rate: int = SAMPLE_RATE) -> list[dict]:
        if len(audio) == 0:
            return []
        frame_len = int(sample_rate * 0.05)
        n_frames = max(1, len(audio) // frame_len)
        trimmed = audio[: n_frames * frame_len]
        frames = trimmed.reshape(n_frames, frame_len)
        rms = np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12)
        
        is_speech = rms > self.threshold_lin
        
        stamps = []
        in_speech = False
        start_frame = 0
        for i, speech in enumerate(is_speech):
            if speech and not in_speech:
                in_speech = True
                start_frame = i
            elif not speech and in_speech:
                in_speech = False
                stamps.append({
                    "start": start_frame * frame_len,
                    "end": i * frame_len
                })
        if in_speech:
            stamps.append({
                "start": start_frame * frame_len,
                "end": len(audio)
            })
        return stamps


class SlidingWindowBuffer:
    """Accumulate mic chunks and emit overlapping speech windows."""

    def __init__(
        self,
        window_sec: float = 1.5,
        step_sec: float = 0.5,
        min_speech_sec: float = 0.5,
        sample_rate: int = SAMPLE_RATE,
    ) -> None:
        self.window_samples = int(window_sec * sample_rate)
        self.step_samples = int(step_sec * sample_rate)
        self.min_speech_samples = int(min_speech_sec * sample_rate)
        self._buffer = np.array([], dtype=np.float32)
        self._total_samples = 0
        self._next_emit = 0

    def push(self, chunk: np.ndarray) -> list[SpeechWindow]:
        if chunk.size == 0:
            return []
        self._buffer = np.concatenate([self._buffer, chunk.astype(np.float32)])
        windows: list[SpeechWindow] = []

        while self._total_samples + len(self._buffer) - self._next_emit >= self.window_samples:
            start = self._next_emit
            end = start + self.window_samples
            if end > self._total_samples + len(self._buffer):
                break
            rel_start = start - self._total_samples
            audio = self._buffer[rel_start : rel_start + self.window_samples].copy()
            windows.append(SpeechWindow(audio=audio, start_sample=start, end_sample=end))
            self._next_emit += self.step_samples

        consumed = self._next_emit - self._total_samples
        if consumed > 0:
            self._buffer = self._buffer[consumed:]
            self._total_samples = self._next_emit

        return [w for w in windows if len(w.audio) >= self.min_speech_samples]

    def iter_windows(self, audio: np.ndarray) -> list[SpeechWindow]:
        """Generate sliding windows over a full waveform."""
        windows: list[SpeechWindow] = []
        offset = 0
        while offset + self.window_samples <= len(audio):
            chunk = audio[offset : offset + self.window_samples]
            windows.append(
                SpeechWindow(
                    audio=chunk.copy(),
                    start_sample=offset,
                    end_sample=offset + self.window_samples,
                )
            )
            offset += self.step_samples
        return [w for w in windows if len(w.audio) >= self.min_speech_samples]


---
### Tải model Speaker Diarization (ERes2Net-Large)

In [ ]:
import subprocess, sys, os
try:
    import numpy
    np_ver = numpy.__version__
except: np_ver = '2.0.2'
print('Installing Rust (required for DeepFilterNet on Python 3.12)...')
os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'addict', 'modelscope', 'deepfilternet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', f'numpy=={np_ver}'], check=True)
import os, re, shutil

TARGET_DIR = '/kaggle/working/SpeakAI-Eval'
ERES_MODEL_DIR = f'{TARGET_DIR}/speaker-diarize/pretrained_models/speech_eres2net_large_200k_sv_zh-cn_16k-common'

if not os.path.exists(f'{ERES_MODEL_DIR}/model.onnx') and not os.path.exists(f'{ERES_MODEL_DIR}/pytorch_model.bin'):
    print('Downloading ERes2Net-Large to Kaggle via Git...')
    if os.path.exists(ERES_MODEL_DIR):
        shutil.rmtree(ERES_MODEL_DIR)
    os.system('git lfs install')
    os.system(f'git clone https://www.modelscope.cn/damo/speech_eres2net_large_200k_sv_zh-cn_16k-common.git {ERES_MODEL_DIR}')
    print('Download done!')
else:
    print('ERes2Net-Large model already exists.')

print('ALL SETUP COMPLETED!')

---
### Tải LLM Qwen, Whisper, WavLM, DeepFilterNet vào Kaggle

In [ ]:
import os
import torch
from huggingface_hub import snapshot_download

TARGET_DIR = '/kaggle/working/SpeakAI-Eval'
WHISPER_DIR = f'{TARGET_DIR}/pretrained_models/whisper-large-v3-turbo'
QWEN_DIR = f'{TARGET_DIR}/pretrained_models/Qwen2.5-3B-Instruct'
WAVLM_DIR = f'{TARGET_DIR}/pretrained_models/wavlm-large'
LANG_ID_DIR = f'{TARGET_DIR}/pretrained_models/lang-id-voxlingua107-ecapa'

print('Downloading DeepFilterNet model (offline caching)...')
import torchaudio
import sys, types
if not hasattr(torchaudio, 'backend'):
    torchaudio.backend = types.ModuleType('torchaudio.backend')
    sys.modules['torchaudio.backend'] = torchaudio.backend
if not hasattr(torchaudio.backend, 'common'):
    torchaudio.backend.common = types.ModuleType('torchaudio.backend.common')
    sys.modules['torchaudio.backend.common'] = torchaudio.backend.common
    torchaudio.backend.common.AudioMetaData = getattr(torchaudio, 'AudioMetaData', type('AudioMetaData', (), {}))
from df.enhance import init_df
init_df()

print('Downloading Whisper model (offline caching)...')
if not os.path.exists(WHISPER_DIR):
    snapshot_download(repo_id='openai/whisper-large-v3-turbo', local_dir=WHISPER_DIR, ignore_patterns=['*.h5', '*.ot', '*.msgpack'], local_dir_use_symlinks=False)
else:
    print('Whisper model already exists.')

print('Downloading WavLM model (offline caching)...')
if not os.path.exists(WAVLM_DIR):
    snapshot_download(repo_id='microsoft/wavlm-large', local_dir=WAVLM_DIR, ignore_patterns=['*.h5', '*.ot', '*.msgpack'], local_dir_use_symlinks=False)
else:
    print('WavLM model already exists.')

print('Downloading Lang-ID model (offline caching)...')
if not os.path.exists(LANG_ID_DIR):
    snapshot_download(repo_id='speechbrain/lang-id-voxlingua107-ecapa', local_dir=LANG_ID_DIR, ignore_patterns=['*.h5', '*.ot', '*.msgpack'], local_dir_use_symlinks=False)
else:
    print('Lang-ID model already exists.')

print('Downloading Qwen LLM (offline caching)...')
if not os.path.exists(QWEN_DIR):
    snapshot_download(repo_id='Qwen/Qwen2.5-3B-Instruct', local_dir=QWEN_DIR, ignore_patterns=['*.h5', '*.ot', '*.msgpack'], local_dir_use_symlinks=False)
else:
    print('Qwen model already exists.')
print('Offline models ready!')
